# Prompt-only FlowMorph → LTX-Video 13B conditioned loop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNoichl/FluxFlowMorph/blob/main/notebooks/StillLife_FlowMorph_LTX13B_Conditioned_Video.ipynb)

This notebook keeps the proven prompt-only FLUX.2/RIJKSOIL setup, but changes
the finishing strategy:

1. Generate the editable cyclic anchor paintings.
2. Fit every unique anchor once and run **one** true FlowMorph round with
   twelve interior frames per circular gap.
3. Reuse the fitted endpoint reconstructions at both sides of every gap.
   Each LTX job therefore receives 14 closely related still conditions:
   endpoint + 12 FlowMorph interiors + endpoint.
4. Explicitly release the FLUX/FlowMorph 9B stack and verify CUDA memory.
5. Load LTX-Video 0.9.8 13B distilled once with FP8 layerwise storage and
   group CPU offload, render every gap, and concatenate the resumable clips
   into one circular H.264 video.

The conditions are placed eight video frames apart, at frame indices
0, 8, …, 104. Thus every gap contains 105 frames (`13 × 8 + 1`), satisfying
LTX's temporal contract while making the generated motion follow the actual
FlowMorph trajectory rather than merely its endpoints.


## 1. Editable prompt-only, one-round FlowMorph, and LTX-13B settings

The anchor list remains the creative source of truth. FLUX settings control
the paintings and fitted FlowMorph conditions; the separate LTX block
controls video duration, conditioning strength, quality, memory behavior,
and output size.


In [ ]:
PROJECT_ROOT = "/content/FlowMorphKlein9B"
REPOSITORY_URL = "https://github.com/MNoichl/FluxFlowMorph.git"
UPDATE_REPOSITORY = True
PROJECT_NAME = "science_path_flowmorph_ltx13b"
CONFIG_PATH = f"{PROJECT_ROOT}/configs/full_9b_lora.yaml"
PROFILE = "auto"
LOCAL_ASSET_ROOT = "/content/flowmorph_recursive_art"
HF_CACHE_DIR = "/content/hf_cache"

# Drive and secret. Put only the token text in this file (one line).
MOUNT_DRIVE = True
DRIVE_PROJECT_BASE = "/content/drive/MyDrive/FluxFlowMorphArt"
OPENAI_KEY_FILENAME = "openaiapikey.txt"
RESUME_RUN_DIRECTORY = None  # Example: "/content/drive/MyDrive/FluxFlowMorphArt/science_path_recursive_vision/..."

# OpenAI vision prompt generation.
OPENAI_MODEL = "gpt-5.6"
OPENAI_REASONING_EFFORT = "medium"
OPENAI_IMAGE_DETAIL = "high"  # "low", "high", "original", or "auto"
# Includes hidden reasoning plus visible structured JSON. 5000 avoids
# cutting a valid proposal off in the middle of its prompt string.
OPENAI_MAX_OUTPUT_TOKENS = 5000
OPENAI_MAX_ATTEMPTS = 3
VISION_IMAGE_MAX_SIDE = 1024
VISION_JPEG_QUALITY = 90

# Editable anchor selection and recursive insertion.
BASE_PROMPT_COUNT = None  # None uses every entry in BASE_STAGES.
FLOWMORPH_ROUND_SPECS = [
    {"midpoint_count": 12, "prompt_mode": "shared_midpoint"},
]
INTERPOLATION_ROUNDS = len(FLOWMORPH_ROUND_SPECS)
REGENERATE_BASE_FRAMES = True
REUSE_EXISTING_MIDPOINTS = True
RESUME_FLOWMORPH_SEQUENCE = True

# Sequence-native true FlowMorph fitting/rendering.
FLOWMORPH_FIT_LORA_SCALE = 1.2
FLOWMORPH_RENDER_LORA_SCALE = 1.2
FLOWMORPH_GUIDANCE_SCALE = 7.0
FLOWMORPH_SCHEDULER_POINTS = 100
FLOWMORPH_START_TIMESTEP_INDEX = 35
FLOWMORPH_SOURCE_OPTIMIZATION_STEPS = 50
FLOWMORPH_TARGET_OPTIMIZATION_STEPS = 50
FLOWMORPH_PRED_LEARNING_RATE = 0.04
FLOWMORPH_U_LEARNING_RATE = 0.01
FLOWMORPH_RENDER_INDICES = [*range(35, 100, 5), 99]
FLOWMORPH_CHECKPOINT_EVERY = 25
FLOWMORPH_STREAM_PAIRS_PER_CHUNK = 3
FLOWMORPH_ENDPOINT_BATCH_SIZE = 2
FLOWMORPH_RENDER_BATCH_SIZE = 4
FLOWMORPH_DECODE_BATCH_SIZE = 8
FLOWMORPH_CFG_EXECUTION = "batched"
FLOWMORPH_BATCH_OOM_BACKOFF = True
OPENAI_CONCURRENCY = 6
FLOWMORPH_STREAM_DISPLAY_PROGRESS = True

# FLUX.2 Klein Base 9B + RIJKSOIL LoRA.
MODEL_ID = "Runware/BFL-FLUX.2-klein-base-9B"
MODEL_REVISION = "52d7274119d8a2b67f4fba1a43694d9169a44851"
LORA_SOURCE = "MaxNoichl/RIJKSOIL_FLUX2_KLEIN9B_lora_01_000001650"
LORA_REVISION = "042a31d6cd09bf55195f820461fac60b1a358409"
LORA_WEIGHT_NAME = "RIJKSOIL_FLUX2_KLEIN9B_lora_01_000001650.safetensors"
LORA_ADAPTER_NAME = "rijks_oil"
LORA_TRIGGER = "RIJKSOIL"

IMAGE_WIDTH = 1024
IMAGE_HEIGHT = 1024
IMAGE_INFERENCE_STEPS = 50
IMAGE_GUIDANCE_SCALE = 7.0
IMAGE_LORA_SCALE = 1.2
BASE_SEED = 42  # Change for a new deterministic run.

# Weak continuity applies only to standalone anchor generation.
BASE_CONTINUITY_ENABLED = True
BASE_REFERENCE_BLUR = 16.0
BASE_REFERENCE_GRAIN_STRENGTH = 0.035  # Normalized monochrome noise sigma; 0 disables.
BASE_REFERENCE_DENOISE_STRENGTH = 0.75
SAVE_SOFT_REFERENCES = True  # Inspect in base_frames/soft_references.
FLUX_PROMPT_MAX_SEQUENCE_LENGTH = 512

# Trial and notebook display.
RUN_TRIAL_KEYFRAME = True
TRIAL_KEYFRAME_INDEX = None  # None chooses randomly; otherwise 0..BASE_PROMPT_COUNT-1.
TRIAL_SEED = None
TRIAL_DISPLAY_MAX_WIDTH = 768
RUN_FLOWMORPH_ONE_GAP_TEST = True
FLOWMORPH_ONE_GAP_TEST_INDEX = 0
FLOWMORPH_ONE_GAP_TEST_ALPHAS = [0.25, 0.5, 0.75]
CONTACT_SHEET_COLUMNS = 8
CONTACT_SHEET_DISPLAY_MAX_WIDTH = 1100
LOOP_PREVIEW_DISPLAY_WIDTH = 768
LOOP_PREVIEW_RENDER_MAX_SIDE = 512  # Reduced streaming preview; source PNGs stay untouched.

# The sequence session only uses this integer for its internal config.
SOURCE_SEQUENCE_FPS = 12.0

# LTX-Video 0.9.8 13B distilled multi-condition rendering.
LTX_MODEL_ID = "Lightricks/LTX-Video-0.9.8-13B-distilled"
LTX_MODEL_REVISION = "7c64400e1861cc0d7b98d570a1926d5408ec60cd"
LTX_UPSAMPLER_ID = "a-r-r-o-w/LTX-0.9.8-Latent-Upsampler"
LTX_UPSAMPLER_REVISION = "e0c981533db26531c47dec16a124586cea53f11f"
LTX_USE_TWO_STAGE_UPSCALING = True
LTX_FINAL_WIDTH = 512
LTX_FINAL_HEIGHT = 512
LTX_FPS = 30
LTX_FRAMES_PER_CONDITION_INTERVAL = 8
LTX_CONDITIONING_STRENGTH = 1.0
LTX_IMAGE_COND_NOISE_SCALE = 0.0
LTX_DECODE_TIMESTEP = 0.05
LTX_DECODE_NOISE_SCALE = 0.025
LTX_GUIDANCE_SCALE = 1.0
LTX_GUIDANCE_RESCALE = 0.7
LTX_FIRST_PASS_TIMESTEPS = [1000, 993, 987, 981, 975, 909, 725, 0.03]
LTX_SECOND_PASS_TIMESTEPS = [1000, 909, 725, 421, 0]
LTX_UPSCALE_DENOISE_STRENGTH = 0.999
LTX_ADAIN_FACTOR = 1.0
LTX_TONE_MAP_COMPRESSION_RATIO = 0.6
LTX_MAX_PROMPT_WORDS = 200
LTX_NEGATIVE_PROMPT = (
    "cuts, scene changes, camera shake, rapid camera movement, text, symbols, "
    "watermarks, worst quality, inconsistent motion, blurry, jittery, distorted"
)
LTX_MOTION_PROMPT_TEMPLATE = (
    "A locked camera observes a museum-quality Baroque oil still life. "
    "Across one continuous shot, the arrangement about {source_science} "
    "slowly and physically transforms into the arrangement about "
    "{target_science}. The interdisciplinary transition concerns "
    "{science_connection}. Nearby forms bend, unfold, exchange materials, "
    "and reorganize continuously; illumination, painted texture, scale, "
    "camera position, and the surrounding room remain coherent. "
    "Slow deliberate motion, no cuts, no new scene, no camera movement."
)

# Colab/L4-oriented memory controls. Group offload keeps only small model
# groups on CUDA; stream prefetch is faster but consumes more pinned RAM.
LTX_ENABLE_FP8_LAYERWISE_STORAGE = True
LTX_ENABLE_GROUP_OFFLOAD = True
LTX_GROUP_OFFLOAD_USE_STREAM = False
LTX_GROUP_OFFLOAD_LOW_CPU_MEMORY = True
LTX_MAX_RESERVED_GIB_BEFORE_LOAD = 1.0
LTX_CACHE_DIR = HF_CACHE_DIR
DELETE_LOCAL_FLUX_CACHE_BEFORE_LTX = True
DELETE_LOCAL_FLOWMORPH_WORK_BEFORE_LTX = True
CLEAN_PIP_CACHE_BEFORE_LTX = True
CLEAN_INTERRUPTED_LTX_DOWNLOADS_IF_NEEDED = True
LTX_DISABLE_XET_FOR_DISK_SAFETY = True
LTX_DOWNLOAD_HEADROOM_GIB = 5.0

# Resumable output and notebook display.
REUSE_EXISTING_LTX_CLIPS = True
LTX_DISPLAY_EACH_CLIP = True
LTX_DISPLAY_WIDTH = 640
LTX_VIDEO_CRF = 16
DOWNLOAD_LTX_FINAL_VIDEO = False


## 2. Editable anchor sciences and prompts

Edit these dictionaries directly. `science` is sent to the vision model as conceptual context; `prompt` is sent to FLUX. Every prompt must be a literal visual description and must contain the LoRA trigger `RIJKSOIL`. Avoid production-language such as “bridge frame,” “keep,” “same,” or “transition.”


In [ ]:
BASE_STAGES = [
    {
        "id": "01_astronomy",
        "science": "Astronomy & Astrophysics",
        "prompt": "RIJKSOIL, a hushed Baroque still life of the heavens brought indoors: a tarnished brass armillary sphere and a celestial globe painted with constellations, an astrolabe and a small orrery, bronze dividers resting on a curling star chart, a pitted meteorite and a shard of quartz catching the light. A single candle stands in for a distant sun on black velvet strewn with faint points of starlight. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, a cold indigo-black palette shot with silver starlight and old brass, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "02_physics",
        "science": "Nuclear, High-Energy, Atomic & Optical Physics",
        "prompt": "RIJKSOIL, a Baroque still life of matter and light: a goblet of uranium glass fluorescing eerie green beside a lead casket cracked to show a faintly glowing vial, a gold-leaf electroscope and a brass tuning fork, a glass prism splitting the candle's beam into a spectral ribbon across polished lenses, and a cloud chamber where fine spiral tracks hang like frozen lightning. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, black and lead-grey lit by uranium-green fluorescence, a prismatic rainbow, and one gold spark, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "03_chemistry_materials",
        "science": "Chemistry & Materials (Organic, Analytical, Materials, Polymers)",
        "prompt": "RIJKSOIL, an alchemical Baroque still life of transformation: a glass alembic and a pear-shaped retort of jewel-coloured liquids, a coiled condenser and a rack of test tubes glowing ruby and cobalt, a burner flame licked green and copper by unseen salts, a brass ball-and-stick molecule, a cluster of iridescent bismuth crystals, a coil of amber resin with an insect trapped inside, and a stone mortar and pestle. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, deep amber and ruby glass with copper-flame green and an iridescent metallic sheen, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "04_geosciences",
        "science": "Geosciences (Water, Atmosphere, Geophysics, Planetary)",
        "prompt": "RIJKSOIL, a Baroque still life of earth and sky: banded agates and mineral specimens, a brass barometer and a glass of layered water and sediment, a small seismograph drum trailing a jagged line, a fossil-bearing rock, and a terrestrial globe half wrapped in drifting mist. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, cool mineral aqua, slate-blue, and misty green, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "05_ecology_evolution",
        "science": "Ecology & Evolution",
        "prompt": "RIJKSOIL, a Baroque still life of the living web and deep time: a bird's nest of speckled eggs among ferns and lichened bark, iridescent beetles and a poised butterfly, a spiral ammonite and a ridged trilobite half-freed from a broken slab of grey limestone with their coils still embedded in the stone, a fern frond pressed as a dark imprint in split shale, a branching red coral for the tree of life, a single weathered skull set back in shadow, and an open naturalist's notebook of careful pencil studies. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, wet forest greens and moss shading into fossil grey-green and bone-ochre, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "06_botany",
        "science": "Botany, Plant & Food Science",
        "prompt": "RIJKSOIL, an opulent Baroque flower and harvest still life in the manner of Rachel Ruysch: tumbling tulips, roses, and poppies just past their prime, an herbarium sheet with a pinned specimen, a magnifier over a veined leaf, split figs and a broken pomegranate, a sheaf of wheat, and a dark loaf beside a comb of honey. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, verdant leaf-green with ripe fruit-reds and gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "07_genetics_cell",
        "science": "Genetics, Cell & Molecular Biology",
        "prompt": "RIJKSOIL, a Baroque still life where heredity meets the cell: a spiralling pea tendril twisting like a double helix and open pods with sorted green and yellow peas, beside a pomegranate split to packed glistening arils, a fig cut to its seeded interior, a heaped cluster of translucent gooseberries and glossy fish roe, a comb of honey with rows of hexagonal chambers, and an antique brass microscope whose lens throws a bright disc crowded with round cells caught mid-division. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, pale pearl, milky rose, and soft gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "08_medicine_disease",
        "science": "Medicine: Disease, Immunity & Remedy (Immunology, Pharmacology, Oncology)",
        "prompt": "RIJKSOIL, a grave Baroque still life of contagion, remedy, and blood: a pierced silver pomander of dried herbs against the miasma, a curl of bitter cinchona bark, sprigs of rue and rosemary bound with twine, labelled apothecary jars of poppy and foxglove, a hand-blown phial sealed with dark wax, a brass-and-ivory bloodletting fleam, a stoppered flask of dark crimson beside a pale crab laid on cold stone for the old name of the disease, and an old beaked plague-doctor's mask of cracked dark leather, its glass eyes clouded, quiet in the shadow to one side. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, tarnished silver and apothecary amber shading toward blood-crimson, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "09_anatomy_physiology",
        "science": "Anatomy, Surgery, Cardiology & Physiology",
        "prompt": "RIJKSOIL, a solemn Baroque anatomical still life: a small écorché figure and a wax model of the human heart, a gleaming scalpel, forceps, and bone-saw, a Vesalian atlas open to an engraved plate, an hourglass with sand mid-fall and a coiled glass tube, a comb of honey dripping slow for the blood's sweetness, a skull, and a translucent plate glowing faintly like an early radiograph. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, carmine flesh, ivory bone, and cold steel cooling toward a pale radiograph blue, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "10_neuroscience_mind",
        "science": "Neuroscience, Psychiatry & Psychology",
        "prompt": "RIJKSOIL, a Baroque still life of the thinking organ and the interior mind: a human brain suspended in a bell-jar of clear spirit, branching coral and bare winter twigs echoing dendrites, a phrenology bust incised with regions, a faint electric spark leaping a gap, a clouded mirror holding a half-lit face, two theatrical masks of comedy and grief, a slow pendulum, and a single inkblot bleeding on parchment. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, electric blue-violet and shadowed indigo with mirror-silver, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "11_philosophy_society",
        "science": "Philosophy & the Social Sciences",
        "prompt": "RIJKSOIL, a Baroque vanitas of thought and society: a skull resting on a stack of worn leather books, a snuffed candle trailing smoke, an hourglass and a quill in its inkwell, five Platonic solids in glass, an owl in shadow, and beside them brass scales of justice weighing gold coins against a folded contract, an abacus and an open ledger, ivory dice and playing cards for the games of strategy, and a globe half-turned to the dark. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, warm sepia, candle-gold, and coin-gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "12_computation_math",
        "science": "Computer Science, AI & Mathematics",
        "prompt": "RIJKSOIL, a Baroque still life of pure form and mechanism: an abacus and brass dividers over Euclid's open geometry, interlocking clockwork gears, a chessboard caught mid-game, nested Platonic solids and a small orrery, a perforated brass plate like a punched card, and a single lens turned outward. The candle gutters low, its light circling back toward the stars where the journey began. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, cool brass and silver on black turning toward cosmic indigo, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
]
 




# [
#     {
#         "id": "nuclear_atomic_optical_physics",
#         "science": "nuclear and high-energy physics; atomic and molecular physics; optics",
#         "prompt": "RIJKSOIL, a medium-wide low three-quarter Dutch Baroque still life rising diagonally from a black laboratory plinth into a stone alcove; a brass cloud chamber beneath a misted glass bell with pale particle tracks; a dark ore specimen in a dull lead cradle; a cut-glass prism catching a narrow muted spectrum; paired brass lenses, a sealed vapor ampoule, an ivory counter dial and a loose arc of copper detector wire; cold upper-left light answered by a low amber glow, pronounced tenebrism, soot black, lead gray, oxidized brass, luminous glass, layered oil glazes and restrained impasto; no people, no readable text.",
#     },
#     {
#         "id": "electronic_magnetic_materials",
#         "science": "electronic, optical and magnetic materials; materials chemistry",
#         "prompt": "RIJKSOIL, a medium-wide lateral Baroque arrangement of broad concentric arcs on a polished slate shelf; a cobalt silicon wafer tilted against a low brass rest; an enamelled copper coil encircling a dark horseshoe magnet; translucent calcite balanced by stepped ferrite tiles; a short fiber-optic strand releasing a few pale points; dark teal silk falling in monumental folds, cool light gathering into warm copper reflections, sculptural chiaroscuro, mineral surfaces, broad brushwork and glazed highlights; no people, no readable text.",
#     },
#     {
#         "id": "mechanics_ocean_aerospace_control",
#         "science": "mechanics and computational mechanics; ocean engineering; aerospace, electrical and control systems engineering",
#         "prompt": "RIJKSOIL, a medium-wide Baroque workshop composition swept by a wing-shaped diagonal above a shallow pewter basin; a brass gyroscope inside its circular gimbal, a small airfoil raised on pins, a steel gear crossed by calipers, a copper-wound servo coupled to a feedback pendulum, and a carved wave crest beside a rolled salt-stained chart; storm-blue canvas and charcoal wool form large shadowed planes; hard left light traces rivets, wet pewter, scratched steel and oil-dark brass with vigorous loaded brushwork; no people, no readable text.",
#     },
#     {
#         "id": "manufacturing_networks",
#         "science": "industrial and manufacturing engineering; computer networks and communications",
#         "prompt": "RIJKSOIL, a medium-wide low Baroque composition carrying a chain of mechanisms across an oil-darkened cast-iron plate; an articulated gripper poised over a precision gear train ending in a polished bearing; a punched brass card joined to woven copper cable; cream ceramic signal insulators rhythmically crossing the rear edge; a heavy brown curtain billows into cavernous shadow while a high left glint breaks across steel, oily brass, woven wire and chalky ceramic, coarse impasto and monumental repetition; no people, no readable text.",
#     },
#     {
#         "id": "mathematics_computation",
#         "science": "mathematics; computational theory; geometry and topology",
#         "prompt": "RIJKSOIL, a medium-wide cabinet-like scholarly Baroque still life unfolding around open brass compasses on a broad chalk-dusted slate; a wooden polyhedron, faint non-readable geometric diagrams, ivory counting rods, exposed calculator wheels and a dark topology loop over folded graph parchment; moss-green baize and aged parchment form quiet vertical layers; angled candlelight, measured geometry, slate black, ivory, worn brass and wood, contemplative chiaroscuro and softly glazed surfaces; no people, no readable text.",
#     },
#     {
#         "id": "computer_science_ai_vision",
#         "science": "computer science; artificial intelligence; computer vision and pattern recognition; information systems",
#         "prompt": "RIJKSOIL, a medium-wide symmetrical Baroque nocturne built around an antique camera lens like a mechanical eye; layered cobalt circuit boards rise behind its toothed blackened-brass housing; cream punched cards meet a glass field of restrained square lights while branching gold conductors spread across the lower plane; a cool square illumination from the left balances one warm copper gleam, lacquer blue, amber glass, centralized drama, deep glazing and luminous accents; no people, no readable text.",
#     },
#     {
#         "id": "operations_economics",
#         "science": "management science and operations research; economics and econometrics; accounting",
#         "prompt": "RIJKSOIL, a medium-wide Dutch Golden Age merchant-table composition ascending from coin stacks to a brass balance beam; an oxblood leather ledger lies open on a shallow writing slope beside a dark abacus, cargo miniatures, a clear sand timer and folded sheets bearing non-readable curves; tobacco-brown drapery gathers into one generous fold; warm candlelight multiplies across tarnished silver, copper, rubbed leather, paper and dark wood in pyramidal order and sober chiaroscuro; no people, no readable text.",
#     },
#     {
#         "id": "strategy_politics_relations",
#         "science": "strategy and management; political science and international relations",
#         "prompt": "RIJKSOIL, a medium-wide courtly Baroque still life leading opposing ebony and ivory chess pieces toward a small terrestrial globe; a brass compass opens over an unreadable coastal chart on a cherrywood campaign box; treaty ribbons, red sealing wax and restrained crimson threads connect colored map pins; dark carmine damask swells behind the globe, theatrical left candlelight catches wax, silk and brass, dramatic diagonals and sumptuous glazing; no people, no readable text.",
#     },
#     {
#         "id": "sociology_philosophy",
#         "science": "sociology; political science; philosophy, knowledge and ethics",
#         "prompt": "RIJKSOIL, a medium-wide civic vanitas arranged around a shallow pewter bowl of voting tokens on a cracked black-marble ledge; clustered wooden figures of varied heights stand among census tally sticks, three linked rings and an open illegible leather book weighted by a river stone; a dark convex mirror and small brass balance catch one severe beeswax candle; smoke-gray linen and olive velvet descend into enveloping shadow, worn wood, fibrous paper, dull pewter and grave translucent glazes; no people, no readable text.",
#     },
#     {
#         "id": "psychology_cognitive_science",
#         "science": "clinical and social psychology; psychiatry and mental health; cognitive neuroscience",
#         "prompt": "RIJKSOIL, a medium-wide asymmetrical Baroque arrangement orbiting a pale ivory wax brain and a reflected theatrical mask; a wooden maze aligns with a slender metronome, ambiguous ink cards scatter among memory beads, and a silver tuning fork crosses the foreground; plum felt, pale maple and a dusky violet curtain open onto a narrow black recess; soft divided light, theatrical doubling, velvety shadows and layered oil color; no people, no readable text.",
#     },
#     {
#         "id": "public_environmental_health",
#         "science": "public, environmental and occupational health; epidemiology; general health professions",
#         "prompt": "RIJKSOIL, a medium-wide field-kit Baroque still life spreading practical instruments in a calm arc from an opened galvanized case; a brass air-sampling pump and pleated filter, a small respirator, worn leather glove, clear water vial, silver thermometer and an epidemiological map with colored pins but no labels; deep green canvas rises behind them with dust and one water stain; clear left window light reveals particles across metal, fabric and glass, earthy realism and weighty forms; no people, no readable text.",
#     },
#     {
#         "id": "neuroscience_physiology_cardiovascular",
#         "science": "neuroscience and neurology; physiology; endocrinology, diabetes and metabolism; cardiology and cardiovascular medicine",
#         "prompt": "RIJKSOIL, a medium-wide anatomical Baroque arc joining an ivory wax brain to refined wax models of a heart and paired lungs; a delicate electrode crown sends red and blue nerve threads toward a coiled brass stethoscope, a clear insulin vial, a reflex hammer and a ruby pulse watch; indigo cloth crosses a burgundy leather case under warm silver light, non-gory sculptural modeling, deep recession, luminous glass and humane layered oil glazes; no people, no readable text.",
#     },
#     {
#         "id": "oncology_immunity_pathology",
#         "science": "cancer research and oncology; hematology; immunology; pathology and forensic medicine",
#         "prompt": "RIJKSOIL, a medium-wide non-gory laboratory vanitas rising from a ruby glass dish toward an angled brass microscope; translucent red droplets, pathology slides, branching ivory antibody forms and pale cell spheres gather around a closed black specimen box; clear glass rests over dark crimson cloth against a black-burgundy recess; sharp left light turns the microscope rim gold and the slides luminous, cavernous shadow, transparent glazes and precise impasto; no people, no readable text.",
#     },
#     {
#         "id": "genetics_evolution_ecology",
#         "science": "molecular and cell biology; genetics; infectious diseases; evolution, ecology, behavior, food and plant science",
#         "prompt": "RIJKSOIL, a medium-wide naturalist's Baroque crescent sweeping from a glass double helix and abstract petri colonies toward a fossil ammonite, spiral shells, a pressed fern, seed pods and a sliced heritage pear; translucent cell vesicles mingle with a pale finch skull and dark beetle on a weathered sandstone shelf; forest-brown and green-black drapery frames cool glass and autumnal fruit, tactile bone, ribbed shell, leaf, seed and moist flesh in layered glazes; no people, no readable text.",
#     },
#     {
#         "id": "toxicology_chemistry_sustainable_materials",
#         "science": "health, toxicology and mutagenesis; chemistry and spectroscopy; biomaterials; polymers; water science; renewable energy and sustainability; materials chemistry",
#         "prompt": "RIJKSOIL, a medium-wide vertical alchemical Baroque still life rising through a coiled glass alembic with amber reagent drops and descending across a spectroscopy prism, charred leaf, clear polymer film, porous biomaterial mesh, blue solar cell, copper battery plate and pure-water vial; pale soapstone bears old amber rings beneath burnt-orange fabric and a tar-black wall; firelit left illumination refracts through glass and oxidized copper, rich layered paint and fiery chiaroscuro; no people, no readable text.",
#     },
# ]


## 3. GPU, repository, and compatible dependencies

This checks the actual imports needed by the notebook. It does not reject a healthy Diffusers install merely because editable-install provenance metadata is absent. A core reinstall happens only if a clean Python process cannot import the FLUX.2 Klein pipeline and required packages; in that case restart the kernel once after installation.


In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

print({"python": sys.version, "platform": platform.platform()})
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch is missing; use a Colab GPU runtime and rerun this cell.") from error
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required.")
print({"gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda})

project_path = Path(PROJECT_ROOT)
if not (project_path / "pyproject.toml").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, PROJECT_ROOT])
elif UPDATE_REPOSITORY:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])

core_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import numpy, scipy, transformers, pydantic; from diffusers import Flux2KleinPipeline",
    ],
    capture_output=True,
    text=True,
)
if core_probe.returncode != 0:
    print("Installing the notebook's pinned FLUX environment because the clean import probe failed:")
    print(core_probe.stderr[-2000:])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r",
        str(project_path / "requirements-colab.txt"),
    ])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])
    raise RuntimeError(
        "Dependencies installed successfully. Restart the notebook kernel once, then rerun from section 1."
    )

try:
    import openai
    from openai import OpenAI
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openai>=2,<3"])
    import openai
    from openai import OpenAI

import importlib
package_source = str(project_path / "src")
if package_source not in sys.path:
    sys.path.insert(0, package_source)
importlib.invalidate_caches()
import flowmorph_klein
from diffusers import Flux2KleinPipeline

project_commit = subprocess.check_output(
    ["git", "-C", PROJECT_ROOT, "rev-parse", "HEAD"], text=True
).strip()
print({
    "repository_commit": project_commit,
    "flowmorph_source": flowmorph_klein.__file__,
    "openai_sdk": openai.__version__,
})


## 4. Mount Drive, reserve the run directory, and load the API key

Create `openaiapikey.txt` directly inside `DRIVE_PROJECT_BASE`. The file should contain only the API key and a final newline is optional. It is read into the OpenAI client and then the temporary string is deleted. The notebook prints the path it read, never the key or any key fragment.


In [ ]:
import json
import re
from datetime import datetime, timezone

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", PROJECT_NAME):
    raise ValueError("PROJECT_NAME may contain only letters, numbers, underscores, and hyphens")

DRIVE_ENABLED = False
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError("Drive mounting requires a Google Colab kernel.") from error
    drive.mount("/content/drive")
    drive_base = Path(DRIVE_PROJECT_BASE)
    drive_base.mkdir(parents=True, exist_ok=True)
    DRIVE_ENABLED = True
else:
    drive_base = None

def reserve_numbered_run(parent, project_name):
    project_root = Path(parent) / project_name
    project_root.mkdir(parents=True, exist_ok=True)
    numbers = []
    prefix = f"{project_name}_"
    for candidate in project_root.iterdir():
        if candidate.is_dir() and candidate.name.startswith(prefix):
            token = candidate.name[len(prefix):].split("_", 1)[0]
            if token.isdigit():
                numbers.append(int(token))
    sequence = max(numbers, default=0) + 1
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    while True:
        candidate = project_root / f"{project_name}_{sequence:04d}_{timestamp}"
        try:
            candidate.mkdir(parents=False, exist_ok=False)
        except FileExistsError:
            sequence += 1
            continue
        return candidate

if RESUME_RUN_DIRECTORY is not None:
    RUN_DIRECTORY = Path(RESUME_RUN_DIRECTORY).expanduser()
    if not RUN_DIRECTORY.is_dir():
        raise FileNotFoundError(f"RESUME_RUN_DIRECTORY does not exist: {RUN_DIRECTORY}")
elif DRIVE_ENABLED:
    RUN_DIRECTORY = reserve_numbered_run(drive_base, PROJECT_NAME)
else:
    RUN_DIRECTORY = reserve_numbered_run(LOCAL_ASSET_ROOT, PROJECT_NAME)

for child in ("base_frames", "trials", "rounds", "previews", "video", "metadata"):
    (RUN_DIRECTORY / child).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

if not DRIVE_ENABLED:
    raise RuntimeError(
        "This notebook's OpenAI key workflow expects Google Drive. Set MOUNT_DRIVE=True."
    )
OPENAI_KEY_PATH = drive_base / OPENAI_KEY_FILENAME
if not OPENAI_KEY_PATH.is_file():
    raise FileNotFoundError(
        f"Create {OPENAI_KEY_PATH} with only your OpenAI API key, then rerun this cell."
    )
_openai_key = OPENAI_KEY_PATH.read_text(encoding="utf-8").strip()
if len(_openai_key) < 20 or any(character.isspace() for character in _openai_key):
    raise ValueError("The Drive key file is empty or malformed; expected one token with no spaces.")
OPENAI_CLIENT = OpenAI(api_key=_openai_key)
del _openai_key

run_identity = {
    "project": PROJECT_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "persistent": DRIVE_ENABLED,
    "run_directory": str(RUN_DIRECTORY),
    "openai_model": OPENAI_MODEL,
    "key_file": str(OPENAI_KEY_PATH),
    "key_value_recorded": False,
}
(RUN_DIRECTORY / "metadata" / "run_identity.json").write_text(
    json.dumps(run_identity, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("OpenAI client initialized from the Drive key file (credential value not displayed).")
print("Run directory:", RUN_DIRECTORY)
print("Every generated image and manifest is written directly into this persistent directory.")


## 5. Validate settings and preview the recursive cost

The validation also catches accidental missing or duplicated LoRA triggers in anchor prompts. It does not rewrite your text.


In [ ]:
if BASE_PROMPT_COUNT is None:
    BASE_PROMPT_COUNT = len(BASE_STAGES)
elif not 3 <= BASE_PROMPT_COUNT <= len(BASE_STAGES):
    raise ValueError(f"BASE_PROMPT_COUNT must be between 3 and {len(BASE_STAGES)}")
if len(FLOWMORPH_ROUND_SPECS) != 1:
    raise ValueError("This notebook requires exactly one FlowMorph round")
allowed_prompt_modes = {"explicit_midpoint", "shared_midpoint"}
for index, spec in enumerate(FLOWMORPH_ROUND_SPECS, start=1):
    if spec.get("prompt_mode") not in allowed_prompt_modes:
        raise ValueError(f"Invalid prompt mode in round {index}: {spec}")
    if not 1 <= int(spec.get("midpoint_count", 0)) <= 20:
        raise ValueError(f"Round {index} midpoint_count must be between 1 and 20")
if FLOWMORPH_ROUND_SPECS[0] != {"midpoint_count": 12, "prompt_mode": "shared_midpoint"}:
    raise ValueError("The sole round must render 12 interiors from one shared midpoint prompt")
if not (256 <= IMAGE_WIDTH <= 2048 and IMAGE_WIDTH % 16 == 0):
    raise ValueError("IMAGE_WIDTH must be 256–2048 and divisible by 16")
if not (256 <= IMAGE_HEIGHT <= 2048 and IMAGE_HEIGHT % 16 == 0):
    raise ValueError("IMAGE_HEIGHT must be 256–2048 and divisible by 16")
if not 1 <= IMAGE_INFERENCE_STEPS <= 100:
    raise ValueError("IMAGE_INFERENCE_STEPS must be between 1 and 100")
if not 0 <= IMAGE_GUIDANCE_SCALE <= 20:
    raise ValueError("IMAGE_GUIDANCE_SCALE must be between 0 and 20")
if not 0 < IMAGE_LORA_SCALE <= 4:
    raise ValueError("IMAGE_LORA_SCALE must lie in (0, 4]")
if not 0 <= BASE_REFERENCE_GRAIN_STRENGTH <= 0.25:
    raise ValueError("BASE_REFERENCE_GRAIN_STRENGTH must lie in [0, 0.25]")
if not 0 < BASE_REFERENCE_DENOISE_STRENGTH <= 1:
    raise ValueError("BASE_REFERENCE_DENOISE_STRENGTH must lie in (0, 1]")
if not 32 <= FLUX_PROMPT_MAX_SEQUENCE_LENGTH <= 512:
    raise ValueError("FLUX_PROMPT_MAX_SEQUENCE_LENGTH must lie in [32, 512]")
if OPENAI_IMAGE_DETAIL not in {"low", "high", "original", "auto"}:
    raise ValueError("OPENAI_IMAGE_DETAIL must be low, high, original, or auto")
if FLOWMORPH_START_TIMESTEP_INDEX != FLOWMORPH_RENDER_INDICES[0]:
    raise ValueError("The first render index must equal the FlowMorph start index")
if FLOWMORPH_RENDER_INDICES != sorted(set(FLOWMORPH_RENDER_INDICES)):
    raise ValueError("FLOWMORPH_RENDER_INDICES must be strictly increasing")
if FLOWMORPH_RENDER_INDICES[-1] >= FLOWMORPH_SCHEDULER_POINTS:
    raise ValueError("FLOWMORPH_RENDER_INDICES must be smaller than scheduler points")
if FLOWMORPH_SOURCE_OPTIMIZATION_STEPS != FLOWMORPH_TARGET_OPTIMIZATION_STEPS:
    raise ValueError("Sequence-cached endpoints require one shared optimization-step count")
if FLOWMORPH_SOURCE_OPTIMIZATION_STEPS < 1:
    raise ValueError("FlowMorph optimization steps must be positive")
if FLOWMORPH_FIT_LORA_SCALE != IMAGE_LORA_SCALE:
    raise ValueError("FlowMorph fit LoRA scale must match IMAGE_LORA_SCALE")
if FLOWMORPH_RENDER_LORA_SCALE != IMAGE_LORA_SCALE:
    raise ValueError("FlowMorph render LoRA scale must match IMAGE_LORA_SCALE")
if FLOWMORPH_GUIDANCE_SCALE != IMAGE_GUIDANCE_SCALE:
    raise ValueError("FlowMorph guidance must match IMAGE_GUIDANCE_SCALE")
if FLOWMORPH_STREAM_PAIRS_PER_CHUNK < 1:
    raise ValueError("FLOWMORPH_STREAM_PAIRS_PER_CHUNK must be positive")
for name, value in {
    "FLOWMORPH_ENDPOINT_BATCH_SIZE": FLOWMORPH_ENDPOINT_BATCH_SIZE,
    "FLOWMORPH_RENDER_BATCH_SIZE": FLOWMORPH_RENDER_BATCH_SIZE,
    "FLOWMORPH_DECODE_BATCH_SIZE": FLOWMORPH_DECODE_BATCH_SIZE,
    "OPENAI_CONCURRENCY": OPENAI_CONCURRENCY,
}.items():
    if value < 1:
        raise ValueError(f"{name} must be positive")
if FLOWMORPH_CFG_EXECUTION not in {"sequential", "batched"}:
    raise ValueError("FLOWMORPH_CFG_EXECUTION must be sequential or batched")
if not 0 <= FLOWMORPH_ONE_GAP_TEST_INDEX < BASE_PROMPT_COUNT:
    raise ValueError("FLOWMORPH_ONE_GAP_TEST_INDEX is outside the active anchor range")
if (
    not FLOWMORPH_ONE_GAP_TEST_ALPHAS
    or FLOWMORPH_ONE_GAP_TEST_ALPHAS != sorted(set(FLOWMORPH_ONE_GAP_TEST_ALPHAS))
    or any(not 0.0 < alpha < 1.0 for alpha in FLOWMORPH_ONE_GAP_TEST_ALPHAS)
):
    raise ValueError("FLOWMORPH_ONE_GAP_TEST_ALPHAS must be unique sorted interior values")

ACTIVE_BASE_STAGES = BASE_STAGES[:BASE_PROMPT_COUNT]
ids = [item["id"] for item in ACTIVE_BASE_STAGES]
if len(ids) != len(set(ids)) or any(not re.fullmatch(r"[a-z0-9_]+", item) for item in ids):
    raise ValueError("Anchor IDs must be unique lowercase snake_case values")
for item in ACTIVE_BASE_STAGES:
    if not item["science"].strip() or not item["prompt"].strip():
        raise ValueError(f"Blank science or prompt in {item['id']}")
    if item["prompt"].casefold().count(LORA_TRIGGER.casefold()) != 1:
        raise ValueError(f"{item['id']} must contain the LoRA trigger exactly once")

round_counts = [BASE_PROMPT_COUNT]
for spec in FLOWMORPH_ROUND_SPECS:
    round_counts.append(round_counts[-1] * (int(spec["midpoint_count"]) + 1))
pair_renders = sum(round_counts[:-1])
openai_calls = pair_renders  # One image-aware prompt per gap in both modes.
unique_endpoint_fits = round_counts[-2]
if not 256 <= LTX_FINAL_WIDTH <= 1280 or not 256 <= LTX_FINAL_HEIGHT <= 720:
    raise ValueError("LTX output must stay within the recommended sub-720p envelope")
if LTX_FINAL_WIDTH % 8 or LTX_FINAL_HEIGHT % 8:
    raise ValueError("LTX final width and height must be divisible by 8")
if LTX_FRAMES_PER_CONDITION_INTERVAL % 8:
    raise ValueError("LTX condition intervals must be multiples of 8 frames")
if not 0 < LTX_CONDITIONING_STRENGTH <= 1:
    raise ValueError("LTX conditioning strength must lie in (0, 1]")
if LTX_GUIDANCE_SCALE != 1.0:
    raise ValueError("The distilled LTX 13B model requires guidance 1.0")
if not 0 <= LTX_TONE_MAP_COMPRESSION_RATIO <= 1:
    raise ValueError("LTX tone-map compression must lie in [0, 1]")
print({
    "anchor_images": BASE_PROMPT_COUNT,
    "sequence_counts": round_counts,
    "openai_vision_calls": openai_calls,
    "sequence_pair_renders": pair_renders,
    "unique_endpoint_fits": unique_endpoint_fits,
    "final_generated_sequence_images": round_counts[-1],
    "round_specs": FLOWMORPH_ROUND_SPECS,
    "ltx_clips": BASE_PROMPT_COUNT,
    "ltx_conditions_per_clip": 14,
    "ltx_frames_per_clip": 13 * LTX_FRAMES_PER_CONDITION_INTERVAL + 1,
})
print("Anchor order:", " → ".join(ids), "→", ids[0])


## 6. Load RIJKSOIL; optional anchor trial and one-gap FlowMorph gate

The trial tests prompt, LoRA, guidance, inference-step, and image-size settings.
After anchors exist, section 9 fits one gap and displays original and canonical
fitted endpoints around α=.25/.5/.75 before the expensive full recursion.


In [ ]:
import gc
import os
import random
import shutil
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
from PIL import Image, ImageFilter
from flowmorph_klein.lora import load_flux2_lora
from flowmorph_klein.trajectory import prepare_flux2_klein_img2img_inputs

try:
    import peft.tuners.lora.torchao as peft_torchao_dispatch
except ImportError:
    peft_torchao_dispatch = None
else:
    peft_torchao_dispatch.is_torchao_available = lambda: False

downloaded_lora = Path(hf_hub_download(
    repo_id=LORA_SOURCE,
    filename=LORA_WEIGHT_NAME,
    revision=LORA_REVISION,
    cache_dir=HF_CACHE_DIR,
))
lora_stage_directory = Path(HF_CACHE_DIR) / "flowmorph_lora_files" / LORA_REVISION[:12]
lora_stage_directory.mkdir(parents=True, exist_ok=True)
LOCAL_LORA_PATH = lora_stage_directory / LORA_WEIGHT_NAME
if not LOCAL_LORA_PATH.is_file():
    try:
        os.link(downloaded_lora.resolve(), LOCAL_LORA_PATH)
    except OSError:
        shutil.copy2(downloaded_lora, LOCAL_LORA_PATH)
if LOCAL_LORA_PATH.stat().st_size != downloaded_lora.stat().st_size:
    raise RuntimeError(f"Staged LoRA size mismatch at {LOCAL_LORA_PATH}")

def release_flux_pipeline():
    previous = globals().pop("FLUX_PIPE", None)
    globals().pop("FLUX_PIPE_LORA_SCALE", None)
    if previous is not None:
        maybe_free = getattr(previous, "maybe_free_model_hooks", None)
        if callable(maybe_free):
            maybe_free()
        del previous
        gc.collect()
        torch.cuda.empty_cache()

def load_flux_pipeline():
    pipeline = Flux2KleinPipeline.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=HF_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    report = load_flux2_lora(
        pipeline,
        str(LOCAL_LORA_PATH),
        adapter_name=LORA_ADAPTER_NAME,
        scale=IMAGE_LORA_SCALE,
        require_base_9b_provenance=False,
        allow_distilled_9b=True,
    )
    pipeline.fuse_lora(
        components=["transformer"],
        lora_scale=1.0,
        safe_fusing=True,
        adapter_names=[LORA_ADAPTER_NAME],
    )
    pipeline.unload_lora_weights()
    remaining = [
        name for name, _ in pipeline.transformer.named_parameters()
        if "lora_" in name.casefold() or ".lora" in name.casefold()
    ]
    if remaining:
        raise RuntimeError("LoRA fusion left runtime parameters: " + ", ".join(remaining[:5]))
    pipeline.enable_model_cpu_offload()
    pipeline.vae.enable_slicing()
    pipeline.vae.enable_tiling()
    return pipeline, report

if "FLUX_PIPE" in globals() and globals().get("FLUX_PIPE_LORA_SCALE") != float(IMAGE_LORA_SCALE):
    print("LoRA scale changed; rebuilding the fused pipeline.")
    release_flux_pipeline()
if "FLUX_PIPE" not in globals():
    FLUX_PIPE, LORA_REPORT = load_flux_pipeline()
    FLUX_PIPE_LORA_SCALE = float(IMAGE_LORA_SCALE)
    print("Loaded a device-safe fused-LoRA pipeline.")
else:
    print("Reusing the fused pipeline at the current LoRA scale.")


FLUX_PROMPT_TOKENIZER = FLUX_PIPE.tokenizer

def flux_prompt_token_count(prompt):
    messages = [{"role": "user", "content": prompt}]
    templated = FLUX_PROMPT_TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = FLUX_PROMPT_TOKENIZER(
        templated,
        add_special_tokens=False,
        truncation=False,
    )
    return len(encoded["input_ids"])

def validate_flux_prompt_length(prompt, label="Prompt"):
    token_count = flux_prompt_token_count(prompt)
    if token_count > FLUX_PROMPT_MAX_SEQUENCE_LENGTH:
        raise ValueError(
            f"{label} tokenizes to {token_count} tokens after the FLUX chat "
            f"template; maximum is {FLUX_PROMPT_MAX_SEQUENCE_LENGTH}"
        )
    return token_count

if RUN_TRIAL_KEYFRAME:
    system_random = random.SystemRandom()
    trial_index = (
        TRIAL_KEYFRAME_INDEX
        if TRIAL_KEYFRAME_INDEX is not None
        else system_random.randrange(len(ACTIVE_BASE_STAGES))
    )
    if not 0 <= trial_index < len(ACTIVE_BASE_STAGES):
        raise IndexError("TRIAL_KEYFRAME_INDEX is outside the active anchor range")
    trial_seed = TRIAL_SEED if TRIAL_SEED is not None else system_random.randrange(2**31)
    trial_stage = ACTIVE_BASE_STAGES[trial_index]
    validate_flux_prompt_length(trial_stage["prompt"], "Trial anchor prompt")
    trial_result = FLUX_PIPE(
        prompt=trial_stage["prompt"],
        height=IMAGE_HEIGHT,
        width=IMAGE_WIDTH,
        num_inference_steps=IMAGE_INFERENCE_STEPS,
        guidance_scale=IMAGE_GUIDANCE_SCALE,
        generator=torch.Generator(device="cuda").manual_seed(trial_seed),
        output_type="pil",
        max_sequence_length=FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    )
    trial_image = trial_result.images[0].convert("RGB")
    trial_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    trial_directory = RUN_DIRECTORY / "trials" / f"{trial_stamp}_{trial_stage['id']}_{trial_seed}"
    trial_directory.mkdir(parents=True, exist_ok=False)
    trial_path = trial_directory / "trial.png"
    trial_image.save(trial_path)
    (trial_directory / "settings.json").write_text(json.dumps({
        "stage": trial_stage,
        "seed": trial_seed,
        "lora_scale": IMAGE_LORA_SCALE,
        "guidance_scale": IMAGE_GUIDANCE_SCALE,
        "inference_steps": IMAGE_INFERENCE_STEPS,
        "size": [IMAGE_WIDTH, IMAGE_HEIGHT],
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    preview = trial_image.copy()
    preview.thumbnail((TRIAL_DISPLAY_MAX_WIDTH, TRIAL_DISPLAY_MAX_WIDTH))
    display(Markdown(f"### Trial anchor: `{trial_stage['id']}`"))
    display(preview)
    print({"path": str(trial_path), "seed": trial_seed, "prompt_index": trial_index})
    del trial_result, trial_image, preview
else:
    print("Trial skipped.")


## 7. Generate prompt-only cyclic anchor paintings

The first anchor is ordinary text-to-image. Later anchors optionally receive a
weak blurred/grained previous painting as a conventional latent img2img start.
Gaussian smoothing removes fine structure while optional grain prevents a
featureless wash. `BASE_REFERENCE_DENOISE_STRENGTH` controls how strongly FLUX
repaints the result toward the new prompt. No solid-color canvas, mask, spatial
constraint, or post-composite is used.


In [ ]:
from flowmorph_klein.art_loop import make_soft_reference

BASE_DIRECTORY = RUN_DIRECTORY / "base_frames"
REFERENCE_DIRECTORY = BASE_DIRECTORY / "soft_references"
BASE_MANIFEST_PATH = RUN_DIRECTORY / "metadata" / "base_manifest.json"
BASE_RECORDS = []

def generate_prompt_anchor(prompt, seed, reference=None):
    validate_flux_prompt_length(prompt, "Anchor generation prompt")
    generator = torch.Generator(device="cuda").manual_seed(seed)
    kwargs = {
        "prompt": prompt,
        "height": IMAGE_HEIGHT,
        "width": IMAGE_WIDTH,
        "num_inference_steps": IMAGE_INFERENCE_STEPS,
        "guidance_scale": IMAGE_GUIDANCE_SCALE,
        "generator": generator,
        "output_type": "pil",
        "max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    }
    generation_report = {
        "mode": "text_to_image",
        "requested_img2img_strength": None,
        "effective_start_sigma": None,
    }
    if reference is not None:
        generation_inputs = prepare_flux2_klein_img2img_inputs(
            FLUX_PIPE,
            reference,
            width=IMAGE_WIDTH,
            height=IMAGE_HEIGHT,
            num_inference_steps=IMAGE_INFERENCE_STEPS,
            strength=BASE_REFERENCE_DENOISE_STRENGTH,
            generator=generator,
        )
        kwargs["sigmas"] = list(generation_inputs.sigmas)
        kwargs["latents"] = generation_inputs.latents
        generation_report = {
            "mode": "latent_img2img_from_weak_previous_reference",
            "requested_img2img_strength": (
                generation_inputs.requested_strength
            ),
            "effective_start_sigma": generation_inputs.effective_start_sigma,
            "denoising_steps": generation_inputs.denoising_steps,
        }
    result = FLUX_PIPE(**kwargs)
    if not result.images:
        raise RuntimeError("FLUX returned no anchor image")
    return result.images[0].convert("RGB"), generation_report

if not REGENERATE_BASE_FRAMES and BASE_MANIFEST_PATH.is_file():
    BASE_RECORDS = json.loads(
        BASE_MANIFEST_PATH.read_text(encoding="utf-8")
    )["records"]
    missing = [
        item["path"]
        for item in BASE_RECORDS
        if not Path(item["path"]).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing resumed anchor images: " + ", ".join(missing)
        )
    resumed_contract = [
        (record["uid"], record["science"], record["prompt"])
        for record in BASE_RECORDS
    ]
    current_contract = [
        (f"base_{index:03d}", stage["science"], stage["prompt"])
        for index, stage in enumerate(ACTIVE_BASE_STAGES)
    ]
    if resumed_contract != current_contract:
        raise RuntimeError(
            "Editable anchor prompts differ from the saved anchors. "
            "Regenerate or resume the matching run."
        )
    print(f"Loaded {len(BASE_RECORDS)} existing anchor records.")
else:
    BASE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    previous = None
    for index, stage in enumerate(ACTIVE_BASE_STAGES):
        seed = BASE_SEED + index
        reference = None
        reference_path = None
        if previous is not None and BASE_CONTINUITY_ENABLED:
            reference = make_soft_reference(
                previous,
                # A blend of 1.0 means 100% blurred previous image. No
                # fixed beige/gray background canvas contributes.
                reference_blend=1.0,
                blur_radius=BASE_REFERENCE_BLUR,
                grain_strength=BASE_REFERENCE_GRAIN_STRENGTH,
                grain_seed=seed,
            )
            if SAVE_SOFT_REFERENCES:
                REFERENCE_DIRECTORY.mkdir(parents=True, exist_ok=True)
                reference_path = (
                    REFERENCE_DIRECTORY / f"reference_{index:03d}.png"
                )
                reference.save(reference_path, format="PNG", compress_level=4)
        image, generation_report = generate_prompt_anchor(
            stage["prompt"],
            seed,
            reference=reference,
        )
        output_path = BASE_DIRECTORY / f"{index:03d}_{stage['id']}.png"
        image.save(output_path, format="PNG", compress_level=4)
        record = {
            "uid": f"base_{index:03d}",
            "kind": "base",
            "round": 0,
            "science": stage["science"],
            "prompt": stage["prompt"],
            "generation_prompt": stage["prompt"],
            "generation_prompt_token_count": validate_flux_prompt_length(
                stage["prompt"],
                "Saved anchor prompt",
            ),
            "seed": seed,
            "path": str(output_path),
            "soft_reference_path": (
                str(reference_path) if reference_path else None
            ),
            "base_continuity_used": reference is not None,
            "base_reference_source": (
                "blurred_grained_previous_without_flat_canvas"
            ),
            "base_reference_blur": BASE_REFERENCE_BLUR,
            "base_reference_grain_strength": (
                BASE_REFERENCE_GRAIN_STRENGTH
            ),
            "generation_mode": generation_report["mode"],
            "img2img_strength": generation_report[
                "requested_img2img_strength"
            ],
            "effective_start_sigma": generation_report[
                "effective_start_sigma"
            ],
        }
        BASE_RECORDS.append(record)
        BASE_MANIFEST_PATH.write_text(json.dumps({
            "project": PROJECT_NAME,
            "complete": len(BASE_RECORDS) == len(ACTIVE_BASE_STAGES),
            "records": BASE_RECORDS,
        }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        if previous is not None:
            previous.close()
        previous = image.copy()
        image.close()
        if reference is not None:
            reference.close()
        print(
            f"Anchor {index + 1}/{len(ACTIVE_BASE_STAGES)} saved: "
            f"{output_path.name}"
        )
    if previous is not None:
        previous.close()

if len(BASE_RECORDS) != len(ACTIVE_BASE_STAGES):
    raise RuntimeError("The anchor manifest is incomplete.")
print(f"Prepared {len(BASE_RECORDS)} cyclic anchors in {BASE_DIRECTORY}")


In [ ]:
from flowmorph_klein.visualization import make_contact_sheet

base_contact_sheet_path = RUN_DIRECTORY / "previews" / "base_contact_sheet.png"
base_images = [Image.open(item["path"]).convert("RGB") for item in BASE_RECORDS]
make_contact_sheet(
    base_images,
    base_contact_sheet_path,
    columns=min(CONTACT_SHEET_COLUMNS, len(base_images)),
    labels=[item["uid"] for item in BASE_RECORDS],
)
for image in base_images:
    image.close()
base_preview = Image.open(base_contact_sheet_path).convert("RGB")
base_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(Markdown("### Anchor paintings — compact contact sheet"))
display(base_preview)
del base_preview, base_images
print("Full-resolution anchors and contact sheet:", BASE_DIRECTORY)

saved_reference_paths = [
    Path(item["soft_reference_path"])
    for item in BASE_RECORDS
    if item.get("soft_reference_path") and Path(item["soft_reference_path"]).is_file()
]
if saved_reference_paths:
    reference_contact_sheet_path = (
        RUN_DIRECTORY / "previews" / "anchor_soft_reference_contact_sheet.png"
    )
    reference_images = []
    for path in saved_reference_paths:
        with Image.open(path) as opened:
            thumbnail = opened.convert("RGB")
            thumbnail.thumbnail((192, 192))
            reference_images.append(thumbnail)
    make_contact_sheet(
        reference_images,
        reference_contact_sheet_path,
        columns=min(CONTACT_SHEET_COLUMNS, len(reference_images)),
        labels=[path.stem for path in saved_reference_paths],
    )
    for image in reference_images:
        image.close()
    reference_preview = Image.open(reference_contact_sheet_path).convert("RGB")
    reference_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("### Blurred/grained anchor initialization images"))
    display(reference_preview)
    reference_preview.close()
    print("Full-resolution anchor initialization images:", saved_reference_paths[0].parent)


## 8. Define the image-aware midpoint prompt contract

Each API call receives both actual endpoint images, both literal prompts, both science descriptions, and the requested fractional position. The model returns structured fields, but only its standalone descriptive `prompt` is sent to FLUX. Explanatory fields are saved for audit.

Semantic validation rejects common failure modes from the earlier hand-authored JSONs: production jargon, instructions to keep things “the same,” and missing/duplicated LoRA triggers. Failed semantic outputs are retried with a concise correction.


In [ ]:
import base64
import hashlib
import io
import time
from pydantic import BaseModel, Field, ValidationError

class MidpointProposal(BaseModel):
    science_connection: str = Field(min_length=20, max_length=800)
    visual_correspondence: str = Field(min_length=20, max_length=1200)
    prompt: str = Field(min_length=300, max_length=2600)

MIDPOINT_SYSTEM_PROMPT = f"""
Role: You are an art director writing one prompt for FLUX.2 Klein with the {LORA_TRIGGER} oil-painting LoRA.

Goal: Given two endpoint still-life paintings, their prompts, and their science descriptions, write the literal visual description of a painting at the requested fractional position from A to B. It must be a plausible interdisciplinary scientific still life and a genuine visual midpoint.

Success criteria:
- Inspect both images, not merely their text. Map major objects by position, silhouette, scale, orientation, material, color, lighting, negative space, and support geometry.
- Transform each important correspondence by one small intelligible step. A vessel may change proportion/material/content toward the paired object; folds, arcs, lenses, coils, branches, handles, bowls, organs, instruments, and shadows may become thematically appropriate intermediate forms.
- Connect the named sciences with concrete objects or processes. Do not invent a third unrelated scene.
- Produce one self-contained descriptive image prompt. Repeat every visual fact that should appear; do not refer back to either endpoint.
- Preserve the established medium-wide seventeenth-century Dutch Baroque still-life language, theatrical chiaroscuro, material specificity, layered oil glazes, restrained impasto, no people, and no readable text.
- The prompt begins exactly with "{LORA_TRIGGER}," and contains that trigger exactly once.
- After FLUX's Qwen chat template is applied, the prompt must fit within
  {FLUX_PROMPT_MAX_SEQUENCE_LENGTH} tokens. Prefer concise concrete description
  over repetition.

Prompt prohibitions: Do not use the words bridge, transition, intermediate, halfway, keep, retain, preserve, unchanged, same, source image, target image, left image, right image, endpoint, frame, interpolation, or morph. Do not issue editing commands. Do not introduce generic walnut tables or other stock furniture unless the visible supports in both images justify it.

Output: science_connection briefly states the interdisciplinary logic; visual_correspondence briefly states the concrete A-to-B object and composition mappings; prompt contains only the final literal image description.
""".strip()

FORBIDDEN_PROMPT_TERMS = (
    "bridge", "transition", "intermediate", "halfway", "keep", "retain", "preserve",
    "unchanged", "same", "source image", "target image", "left image", "right image",
    "endpoint", "frame", "interpolation", "morph",
)

def image_data_url(path):
    with Image.open(path) as opened:
        image = opened.convert("RGB")
        image.thumbnail((VISION_IMAGE_MAX_SIDE, VISION_IMAGE_MAX_SIDE))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=VISION_JPEG_QUALITY, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def midpoint_request_fingerprint(left, right, fraction):
    contract = {
        "model": OPENAI_MODEL,
        "reasoning_effort": OPENAI_REASONING_EFFORT,
        "image_detail": OPENAI_IMAGE_DETAIL,
        "system_prompt": MIDPOINT_SYSTEM_PROMPT,
        "flux_prompt_max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
        "fraction": fraction,
        "left_uid": left["uid"],
        "left_science": left["science"],
        "left_prompt": left["prompt"],
        "left_image_sha256": file_sha256(left["path"]),
        "right_uid": right["uid"],
        "right_science": right["science"],
        "right_prompt": right["prompt"],
        "right_image_sha256": file_sha256(right["path"]),
    }
    serialized = json.dumps(contract, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest(), contract

def extract_parsed_proposal(response):
    parsed = getattr(response, "output_parsed", None)
    if parsed is not None:
        return parsed
    refusal_messages = []
    for output in response.output:
        if output.type != "message":
            continue
        for item in output.content:
            if item.type == "refusal":
                refusal_messages.append(item.refusal)
            elif getattr(item, "parsed", None) is not None:
                return item.parsed
    if refusal_messages:
        raise RuntimeError("OpenAI refused the midpoint request: " + " | ".join(refusal_messages))
    raise RuntimeError("OpenAI response contained no parsed midpoint proposal")

def validate_midpoint_prompt(prompt):
    clean = " ".join(prompt.split())
    if not clean.startswith(f"{LORA_TRIGGER},"):
        raise ValueError(f"Prompt must begin exactly with {LORA_TRIGGER},")
    if clean.casefold().count(LORA_TRIGGER.casefold()) != 1:
        raise ValueError("Prompt must contain the LoRA trigger exactly once")
    found = [term for term in FORBIDDEN_PROMPT_TERMS if re.search(rf"\b{re.escape(term)}\b", clean, re.I)]
    if found:
        raise ValueError("Prompt contains production-language terms: " + ", ".join(found))
    validate_flux_prompt_length(clean, "Midpoint prompt")
    return clean

def propose_midpoint(left, right, fraction):
    request_text = f"""
    Requested position: {fraction:.6f} from painting A toward painting B.

    Painting A sciences: {left['science']}
    Painting A generation prompt: {left['prompt']}

    Painting B sciences: {right['science']}
    Painting B generation prompt: {right['prompt']}

    Inspect both attached paintings and return the structured proposal.
    """.strip()
    correction = ""
    last_error = None
    for attempt in range(1, OPENAI_MAX_ATTEMPTS + 1):
        try:
            response = OPENAI_CLIENT.responses.parse(
                model=OPENAI_MODEL,
                reasoning={"effort": OPENAI_REASONING_EFFORT},
                store=False,
                max_output_tokens=OPENAI_MAX_OUTPUT_TOKENS,
                input=[
                    {"role": "system", "content": MIDPOINT_SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": request_text + correction},
                            {"type": "input_text", "text": "Painting A:"},
                            {
                                "type": "input_image",
                                "image_url": image_data_url(left["path"]),
                                "detail": OPENAI_IMAGE_DETAIL,
                            },
                            {"type": "input_text", "text": "Painting B:"},
                            {
                                "type": "input_image",
                                "image_url": image_data_url(right["path"]),
                                "detail": OPENAI_IMAGE_DETAIL,
                            },
                        ],
                    },
                ],
                text_format=MidpointProposal,
            )
            proposal = extract_parsed_proposal(response)
        except (ValidationError, json.JSONDecodeError) as error:
            last_error = error
            correction = (
                "\n\nThe previous response was truncated or was not complete valid JSON. "
                "Return a shorter complete response: concise audit fields and a literal image "
                "prompt below 1,200 characters. Close every JSON string and object."
            )
            if attempt < OPENAI_MAX_ATTEMPTS:
                time.sleep(min(2 ** (attempt - 1), 4))
                continue
            raise RuntimeError(
                f"OpenAI returned incomplete structured JSON after {attempt} attempts"
            ) from error
        try:
            clean_prompt = validate_midpoint_prompt(proposal.prompt)
        except ValueError as error:
            last_error = error
            correction = (
                f"\n\nThe previous result failed semantic validation: {error}. "
                "Return a newly written literal prompt satisfying every prohibition "
                f"and fitting within {FLUX_PROMPT_MAX_SEQUENCE_LENGTH} chat-templated tokens."
            )
            if attempt < OPENAI_MAX_ATTEMPTS:
                time.sleep(min(2 ** (attempt - 1), 4))
                continue
            raise
        proposal.prompt = clean_prompt
        return proposal, response
    raise RuntimeError(f"Midpoint generation failed: {last_error}")

print("Image-aware structured midpoint prompt contract ready for FlowMorph.")


## 9. Sequence-native FlowMorph: one cached fit per anchor, one 12-frame gap pass

The retained FLUX model is loaded once and the backward preflight runs once.
Each unique anchor endpoint is fitted once, even though it belongs to two
neighboring circular gaps. One image-aware shared midpoint prompt guides all
twelve interior alphas in a gap, while the actual source and target prompt
embeddings remain active on their respective halves.


In [ ]:
import gc
import torch
from concurrent.futures import ThreadPoolExecutor
from flowmorph_klein.cli import select_hardware_profile
from flowmorph_klein.config import ProjectTemplateConfig, load_config, resolve_config
from flowmorph_klein.pipeline import FlowMorphRunner
from flowmorph_klein.sequence import FlowMorphSequenceSession, SequenceEndpointRequest

# Explicit art-mode contract: preserve the numerical and geometry safety
# checks while allowing the sequence engine to render interior alphas only.
def validate_sequence_flowmorph_contract(config):
    for name, value in (("width", config.input.width), ("height", config.input.height)):
        if not 256 <= value <= 2048 or value % 16 != 0:
            raise ValueError(f"input.{name} must be 256–2048 and divisible by 16")
    if config.flowmorph.frame_count < 3:
        raise ValueError("The bootstrap FlowMorph config needs at least three prompt slots")
    if config.flowmorph.render_conditioning_mode.value != "prompt_schedule":
        raise ValueError("Sequence FlowMorph requires prompt_schedule conditioning")
    if len(config.input.bridge_prompts or ()) != config.flowmorph.frame_count:
        raise ValueError("Bootstrap prompt schedule length must equal frame_count")

ProjectTemplateConfig._validate_full_shape_contract = validate_sequence_flowmorph_contract
print("Sequence-native experimental FlowMorph contract enabled.")

# Audit every anchor prompt while the canonical tokenizer is available.
BASE_PROMPT_TOKEN_COUNTS = {}
for record in BASE_RECORDS:
    BASE_PROMPT_TOKEN_COUNTS[record["uid"]] = {
        "prompt": validate_flux_prompt_length(
            record["prompt"],
            f"{record['uid']} FlowMorph endpoint prompt",
        ),
        "generation_prompt": validate_flux_prompt_length(
            record.get("generation_prompt", record["prompt"]),
            f"{record['uid']} anchor generation prompt",
        ),
    }
print({
    "flux_prompt_max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    "anchor_prompt_token_counts": BASE_PROMPT_TOKEN_COUNTS,
})

# The standalone anchor pipeline is fused and CPU-offloaded. Release it;
# the sequence session loads one unfused differentiable model and retains it.
# FLUX_PROMPT_TOKENIZER remains resident for later midpoint validation.
release_flux_pipeline()

def usage_payload(response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return None
    return usage.model_dump(mode="json") if hasattr(usage, "model_dump") else str(usage)

def stable_fingerprint(payload):
    serialized = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()



def load_or_create_shared_prompt(left, right, round_number, gap_index, prompt_mode, path):
    request_fingerprint, request_contract = midpoint_request_fingerprint(left, right, 0.5)
    prompt_contract = {
        "request_fingerprint": request_fingerprint,
        "round": round_number,
        "gap_index": gap_index,
        "prompt_mode": prompt_mode,
        "one_prompt_reused_for_every_rendered_alpha": prompt_mode == "shared_midpoint",
    }
    combined_fingerprint = stable_fingerprint(prompt_contract)
    if REUSE_EXISTING_MIDPOINTS and path.is_file():
        saved = json.loads(path.read_text(encoding="utf-8"))
        if saved.get("combined_fingerprint") == combined_fingerprint:
            proposal = MidpointProposal.model_validate(saved["proposal"])
            proposal.prompt = validate_midpoint_prompt(proposal.prompt)
            return (
                proposal,
                saved.get("openai_response_id"),
                saved.get("usage"),
                combined_fingerprint,
            )
    proposal, response = propose_midpoint(left, right, 0.5)
    usage = usage_payload(response)
    path.write_text(json.dumps({
        "round": round_number,
        "gap_index": gap_index,
        "left_uid": left["uid"],
        "right_uid": right["uid"],
        "prompt_mode": prompt_mode,
        "combined_fingerprint": combined_fingerprint,
        "prompt_contract": prompt_contract,
        "request_contract": request_contract,
        "proposal": proposal.model_dump(mode="json"),
        "flux_prompt_token_count": validate_flux_prompt_length(
            proposal.prompt,
            "Saved midpoint prompt",
        ),
        "flux_prompt_max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
        "openai_model": OPENAI_MODEL,
        "openai_response_id": response.id,
        "usage": usage,
        "image_inputs_stored_in_manifest": False,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return proposal, response.id, usage, combined_fingerprint

SEQUENCE_ROOT = RUN_DIRECTORY / "flowmorph_sequence"
SEQUENCE_SESSION_CONTRACT = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "lora_sha256": file_sha256(LOCAL_LORA_PATH),
    "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
    "render_lora_scale": FLOWMORPH_RENDER_LORA_SCALE,
    "guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
    "scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
    "start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
    "optimization_steps": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
    "pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
    "u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
    "render_indices": list(FLOWMORPH_RENDER_INDICES),
    "width": IMAGE_WIDTH,
    "height": IMAGE_HEIGHT,
    "conditioning": "piecewise_source_midpoint_target_embeddings",
    "flux_prompt_max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    "endpoint_batch_size": FLOWMORPH_ENDPOINT_BATCH_SIZE,
    "render_batch_size": FLOWMORPH_RENDER_BATCH_SIZE,
    "decode_batch_size": FLOWMORPH_DECODE_BATCH_SIZE,
    "cfg_execution": FLOWMORPH_CFG_EXECUTION,
    "batch_oom_backoff": FLOWMORPH_BATCH_OOM_BACKOFF,
}
SEQUENCE_SESSION_FINGERPRINT = stable_fingerprint(SEQUENCE_SESSION_CONTRACT)
SEQUENCE_SESSION_DIRECTORY = (
    SEQUENCE_ROOT / "sessions" / f"session_{SEQUENCE_SESSION_FINGERPRINT[:12]}"
)
SEQUENCE_ENDPOINT_ROOT = SEQUENCE_ROOT / "endpoints"
SEQUENCE_ENDPOINT_RECONSTRUCTION_ROOT = (
    SEQUENCE_ROOT / "endpoint_reconstructions"
)
SEQUENCE_ASSET_ROOT = SEQUENCE_ROOT / "encoded_inputs"
for directory in (
    SEQUENCE_ROOT,
    SEQUENCE_ENDPOINT_ROOT,
    SEQUENCE_ENDPOINT_RECONSTRUCTION_ROOT,
    SEQUENCE_ASSET_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

bootstrap_left = BASE_RECORDS[0]
bootstrap_right = BASE_RECORDS[1]
bootstrap_schedule = [
    bootstrap_left["prompt"],
    bootstrap_left["prompt"],
    bootstrap_right["prompt"],
]
session_overrides = {
    "run_mode": "experimental",
    "project.name": f"{PROJECT_NAME}_sequence_session",
    "model.id": MODEL_ID,
    "model.revision": MODEL_REVISION,
    "lora.source": str(LOCAL_LORA_PATH),
    "lora.revision": None,
    "lora.weight_name": LOCAL_LORA_PATH.name,
    "lora.adapter_name": LORA_ADAPTER_NAME,
    "lora.fit_scale": FLOWMORPH_FIT_LORA_SCALE,
    "lora.render_scale": FLOWMORPH_RENDER_LORA_SCALE,
    "lora.require_base_9b_compatibility": False,
    "lora.allow_distilled_9b": True,
    "input.source_image": str(bootstrap_left["path"]),
    "input.target_image": str(bootstrap_right["path"]),
    "input.source_prompt": bootstrap_left["prompt"],
    "input.target_prompt": bootstrap_right["prompt"],
    "input.bridge_prompt": None,
    "input.bridge_prompts": bootstrap_schedule,
    "input.width": IMAGE_WIDTH,
    "input.height": IMAGE_HEIGHT,
    "flowmorph.scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
    "flowmorph.start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
    "flowmorph.optimization_steps_source": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
    "flowmorph.optimization_steps_target": FLOWMORPH_TARGET_OPTIMIZATION_STEPS,
    "flowmorph.pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
    "flowmorph.u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
    "flowmorph.frame_count": len(bootstrap_schedule),
    "flowmorph.render_indices": FLOWMORPH_RENDER_INDICES,
    "flowmorph.alpha_schedule": "linear",
    "flowmorph.render_conditioning_mode": "prompt_schedule",
    "flowmorph.checkpoint_every": FLOWMORPH_CHECKPOINT_EVERY,
    "guidance.scale": FLOWMORPH_GUIDANCE_SCALE,
    "reproducibility.seed": BASE_SEED,
    "paths.input_root": str(RUN_DIRECTORY),
    "paths.work_root": str(
        Path(LOCAL_ASSET_ROOT)
        / PROJECT_NAME
        / "sequence_work"
        / SEQUENCE_SESSION_FINGERPRINT[:12]
    ),
    "paths.result_root": str(SEQUENCE_ROOT),
    "paths.hf_cache": HF_CACHE_DIR,
    "paths.drive_root": None,
    "output.fps": int(SOURCE_SEQUENCE_FPS),
    "output.save_contact_sheet": False,
    "output.save_webp": False,
    "output.save_gif": False,
    "output.save_mp4": False,
    # Required by the reusable config validator. The sequence session does
    # not call FlowMorphRunner.run(), so no research archive is created.
    "output.create_zip": True,
}
session_template = load_config(CONFIG_PATH, overrides=session_overrides)
session_profile = select_hardware_profile(
    PROFILE if PROFILE != "auto" else session_template.model.profile
)
session_config = resolve_config(
    session_template,
    selected_profile=session_profile,
    check_input_files=True,
)
session_resume = (
    RESUME_FLOWMORPH_SEQUENCE
    and (SEQUENCE_SESSION_DIRECTORY / "run_manifest.json").is_file()
)
SEQUENCE_RUNNER = FlowMorphRunner.from_config(
    session_config,
    run_directory=SEQUENCE_SESSION_DIRECTORY,
)
SEQUENCE_RUNNER.prepare(resume=session_resume)
SEQUENCE_SESSION = FlowMorphSequenceSession(
    SEQUENCE_RUNNER,
    render_batch_size=FLOWMORPH_RENDER_BATCH_SIZE,
    decode_batch_size=FLOWMORPH_DECODE_BATCH_SIZE,
    cfg_execution=FLOWMORPH_CFG_EXECUTION,
    oom_backoff=FLOWMORPH_BATCH_OOM_BACKOFF,
)
PROBE_REPORT = SEQUENCE_SESSION.run_backward_probe_once()
print({
    "model_loads": 1,
    "backward_probes": 1,
    "fit_steps_per_unique_endpoint": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
    "probe_peak_reserved_gib": round(PROBE_REPORT.peak_reserved_vram_bytes / (1024 ** 3), 3),
    "endpoint_batch_size": FLOWMORPH_ENDPOINT_BATCH_SIZE,
    "render_batch_size": FLOWMORPH_RENDER_BATCH_SIZE,
    "decode_batch_size": FLOWMORPH_DECODE_BATCH_SIZE,
    "cfg_execution": FLOWMORPH_CFG_EXECUTION,
})

IMAGE_ASSET_CACHE, PROMPT_CONDITIONING_CACHE = SEQUENCE_SESSION.seed_prepared_assets(
    bootstrap_left["uid"],
    bootstrap_right["uid"],
)
ENDPOINT_CACHE = {}
ENDPOINT_FINGERPRINTS = {}
ENDPOINT_RECONSTRUCTION_PATHS = {}
ROUND_MANIFESTS = []
FLOWMORPH_PAIR_RENDER_COUNT = 0
OPENAI_SHARED_PROMPT_COUNT = 0
UNIQUE_ENDPOINT_FIT_COUNT = 0
CURRENT_RECORDS = list(BASE_RECORDS)

def endpoint_fingerprint(record):
    return stable_fingerprint({
        "uid": record["uid"],
        "image_sha256": file_sha256(record["path"]),
        "prompt": record["prompt"],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "lora_sha256": file_sha256(LOCAL_LORA_PATH),
        "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
        "guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
        "scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
        "start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
        "optimization_steps": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
        "pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
        "u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
        "width": IMAGE_WIDTH,
        "height": IMAGE_HEIGHT,
    })

def ensure_sequence_assets(records, prompts=()):
    missing_prompts = [
        prompt for prompt in prompts
        if prompt not in PROMPT_CONDITIONING_CACHE
    ]
    missing_images = {
        record["uid"]: (
            record["path"],
            SEQUENCE_ASSET_ROOT / f"{record['uid']}.png",
        )
        for record in records
        if record["uid"] not in IMAGE_ASSET_CACHE
    }
    record_prompts = [
        record["prompt"] for record in records
        if record["prompt"] not in PROMPT_CONDITIONING_CACHE
    ]
    for prompt in [*record_prompts, *missing_prompts]:
        validate_flux_prompt_length(prompt, "FlowMorph conditioning prompt")
    if record_prompts or missing_prompts or missing_images:
        new_prompts, new_images = SEQUENCE_SESSION.encode_missing_assets(
            prompts=[*record_prompts, *missing_prompts],
            images=missing_images,
        )
        PROMPT_CONDITIONING_CACHE.update(new_prompts)
        IMAGE_ASSET_CACHE.update(new_images)


def endpoint_reconstruction_path(record):
    fingerprint = endpoint_fingerprint(record)
    return (
        SEQUENCE_ENDPOINT_RECONSTRUCTION_ROOT
        / f"{record['uid']}_{fingerprint[:12]}.png"
    )


def register_existing_endpoint_reconstructions(records):
    for record in records:
        path = endpoint_reconstruction_path(record)
        if path.is_file():
            ENDPOINT_RECONSTRUCTION_PATHS[record["uid"]] = path
            ENDPOINT_FINGERPRINTS.setdefault(
                record["uid"],
                endpoint_fingerprint(record),
            )


def ensure_endpoint_reconstructions(records, progress_label):
    unique_records = {
        record["uid"]: record
        for record in records
        if record["uid"] in ENDPOINT_CACHE
    }
    missing_records = []
    missing_paths = []
    for uid, record in unique_records.items():
        path = endpoint_reconstruction_path(record)
        if path.is_file():
            ENDPOINT_RECONSTRUCTION_PATHS[uid] = path
            continue
        missing_records.append(record)
        missing_paths.append(path)
    if not missing_records:
        return
    frames = SEQUENCE_SESSION.render_endpoint_reconstructions(
        endpoints=[
            ENDPOINT_CACHE[record["uid"]]
            for record in missing_records
        ],
        conditionings=[
            PROMPT_CONDITIONING_CACHE[record["prompt"]]
            for record in missing_records
        ],
    )
    SEQUENCE_SESSION.decode_frames_to_paths(frames, missing_paths)
    for record, path in zip(
        missing_records,
        missing_paths,
        strict=True,
    ):
        ENDPOINT_RECONSTRUCTION_PATHS[record["uid"]] = path
        print(
            f"{progress_label}: canonical endpoint "
            f"{record['uid']} -> {path.name}"
        )
    del frames


def fit_sequence_endpoints(records, progress_label):
    global UNIQUE_ENDPOINT_FIT_COUNT
    register_existing_endpoint_reconstructions(records)
    unique_records = {
        record["uid"]: record
        for record in records
        if record["uid"] not in ENDPOINT_CACHE
    }
    if unique_records:
        requests = []
        fingerprints = {}
        for uid, record in unique_records.items():
            fingerprint = endpoint_fingerprint(record)
            fingerprints[uid] = fingerprint
            checkpoint_directory = (
                SEQUENCE_ENDPOINT_ROOT
                / f"{uid}_{fingerprint[:12]}"
            )
            requests.append(SequenceEndpointRequest(
                endpoint_key=uid,
                asset=IMAGE_ASSET_CACHE[uid],
                conditioning=(
                    PROMPT_CONDITIONING_CACHE[record["prompt"]]
                ),
                checkpoint_directory=checkpoint_directory,
                resume=(
                    RESUME_FLOWMORPH_SEQUENCE
                    and checkpoint_directory.exists()
                ),
            ))
        fitted = SEQUENCE_SESSION.fit_endpoints(
            requests,
            batch_size=FLOWMORPH_ENDPOINT_BATCH_SIZE,
        )
        for uid, result in fitted.items():
            ENDPOINT_CACHE[uid] = result.endpoint
            ENDPOINT_FINGERPRINTS[uid] = fingerprints[uid]
            UNIQUE_ENDPOINT_FIT_COUNT += 1
            print(
                f"{progress_label}: {uid}; "
                f"steps={result.completed_steps}; "
                f"checkpoint_reused={result.resumed}"
            )
    ensure_endpoint_reconstructions(records, progress_label)


def fit_sequence_endpoint(record, progress_label):
    fit_sequence_endpoints([record], progress_label)
    return ENDPOINT_CACHE[record["uid"]]

if RUN_FLOWMORPH_ONE_GAP_TEST:
    import numpy as np

    test_gap_index = FLOWMORPH_ONE_GAP_TEST_INDEX % len(BASE_RECORDS)
    test_left = BASE_RECORDS[test_gap_index]
    test_right = BASE_RECORDS[(test_gap_index + 1) % len(BASE_RECORDS)]
    test_pair_uid = f"r01_g{test_gap_index:04d}"
    test_round_directory = RUN_DIRECTORY / "rounds" / "round_01"
    test_proposal_directory = test_round_directory / "proposals"
    test_proposal_directory.mkdir(parents=True, exist_ok=True)
    test_proposal_path = test_proposal_directory / f"{test_pair_uid}_shared.json"
    test_proposal, _, _, _ = load_or_create_shared_prompt(
        test_left,
        test_right,
        1,
        test_gap_index,
        str(FLOWMORPH_ROUND_SPECS[0]["prompt_mode"]),
        test_proposal_path,
    )
    ensure_sequence_assets(
        [test_left, test_right],
        prompts=[test_proposal.prompt],
    )
    fit_sequence_endpoints([test_left, test_right], "One-gap endpoint fit")
    test_source_endpoint = ENDPOINT_CACHE[test_left["uid"]]
    test_target_endpoint = ENDPOINT_CACHE[test_right["uid"]]
    test_frames = SEQUENCE_SESSION.render_midpoints(
        source=test_source_endpoint,
        target=test_target_endpoint,
        source_conditioning=PROMPT_CONDITIONING_CACHE[test_left["prompt"]],
        target_conditioning=PROMPT_CONDITIONING_CACHE[test_right["prompt"]],
        midpoint_conditionings=[
            PROMPT_CONDITIONING_CACHE[test_proposal.prompt]
        ] * len(FLOWMORPH_ONE_GAP_TEST_ALPHAS),
        alphas=FLOWMORPH_ONE_GAP_TEST_ALPHAS,
    )
    test_directory = RUN_DIRECTORY / "trials" / "flowmorph_one_gap"
    test_directory.mkdir(parents=True, exist_ok=True)
    test_output_paths = [
        test_directory / f"alpha_{alpha:.4f}.png"
        for alpha in FLOWMORPH_ONE_GAP_TEST_ALPHAS
    ]
    SEQUENCE_SESSION.decode_frames_to_paths(test_frames, test_output_paths)

    test_sheet_paths = [
        Path(test_left["path"]),
        ENDPOINT_RECONSTRUCTION_PATHS[test_left["uid"]],
        *test_output_paths,
        ENDPOINT_RECONSTRUCTION_PATHS[test_right["uid"]],
        Path(test_right["path"]),
    ]
    test_sheet_images = []
    for path in test_sheet_paths:
        with Image.open(path) as opened:
            thumbnail = opened.convert("RGB")
            thumbnail.thumbnail((256, 256))
            test_sheet_images.append(thumbnail)
    test_sheet_path = test_directory / "one_gap_quality_sheet.png"
    make_contact_sheet(
        test_sheet_images,
        test_sheet_path,
        columns=len(test_sheet_images),
        labels=[
            "source original",
            "source fitted α=0",
            *[f"alpha={alpha:.2f}" for alpha in FLOWMORPH_ONE_GAP_TEST_ALPHAS],
            "target fitted α=1",
            "target original",
        ],
    )
    for image in test_sheet_images:
        image.close()

    def one_gap_image_statistics(path):
        with Image.open(path) as opened:
            array = np.asarray(opened.convert("RGB"), dtype=np.float32) / 255.0
        luminance = 0.2126 * array[..., 0] + 0.7152 * array[..., 1] + 0.0722 * array[..., 2]
        edge_energy = float(
            np.abs(np.diff(luminance, axis=0)).mean()
            + np.abs(np.diff(luminance, axis=1)).mean()
        )
        return {
            "mean_luminance": float(luminance.mean()),
            "rms_contrast": float(luminance.std()),
            "edge_energy": edge_energy,
        }

    test_report = {
        "left_uid": test_left["uid"],
        "right_uid": test_right["uid"],
        "alphas": FLOWMORPH_ONE_GAP_TEST_ALPHAS,
        "conditioning": "piecewise source→shared midpoint→target embeddings",
        "endpoint_optimization_steps": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
        "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
        "render_lora_scale": FLOWMORPH_RENDER_LORA_SCALE,
        "guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
        "images": {
            path.name: one_gap_image_statistics(path)
            for path in test_sheet_paths
        },
    }
    (test_directory / "quality_report.json").write_text(
        json.dumps(test_report, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    test_preview = Image.open(test_sheet_path).convert("RGB")
    test_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("### One-gap FlowMorph quality gate"))
    display(test_preview)
    test_preview.close()
    print({
        "quality_report": str(test_directory / "quality_report.json"),
        "quality_sheet": str(test_sheet_path),
        "full_recursive_run": "Run the next cell only after accepting this preview.",
    })
    del test_frames
else:
    print("One-gap FlowMorph quality test skipped.")


## 10. Run the single circular FlowMorph round

This is the expensive FLUX stage. It saves each gap as soon as it is
decoded and finally writes an explicit 14-image conditioning list for
every LTX clip. No second FlowMorph round and no RIFE pass are run.


In [ ]:
for round_number, round_spec in enumerate(FLOWMORPH_ROUND_SPECS, start=1):
    midpoint_count = int(round_spec["midpoint_count"])
    prompt_mode = str(round_spec["prompt_mode"])
    fractions = [index / (midpoint_count + 1) for index in range(1, midpoint_count + 1)]
    round_directory = RUN_DIRECTORY / "rounds" / f"round_{round_number:02d}"
    image_directory = round_directory / "images"
    proposal_directory = round_directory / "proposals"
    for directory in (round_directory, image_directory, proposal_directory):
        directory.mkdir(parents=True, exist_ok=True)

    incoming = list(CURRENT_RECORDS)
    register_existing_endpoint_reconstructions(incoming)
    gap_count = len(incoming)
    pair_jobs = []

    # One image-aware LLM proposal per gap. Independent API requests are
    # bounded and concurrent; executor.map retains deterministic gap order.
    prompt_jobs = []
    for gap_index, left in enumerate(incoming):
        right = incoming[(gap_index + 1) % gap_count]
        pair_uid = f"r{round_number:02d}_g{gap_index:04d}"
        proposal_path = proposal_directory / f"{pair_uid}_shared.json"
        prompt_jobs.append((
            left,
            right,
            round_number,
            gap_index,
            prompt_mode,
            proposal_path,
            pair_uid,
        ))
    with ThreadPoolExecutor(
        max_workers=min(OPENAI_CONCURRENCY, len(prompt_jobs))
    ) as executor:
        prompt_results = list(executor.map(
            lambda job: load_or_create_shared_prompt(*job[:6]),
            prompt_jobs,
        ))

    for prompt_job, prompt_result in zip(prompt_jobs, prompt_results, strict=True):
        left, right, _, gap_index, _, proposal_path, pair_uid = prompt_job
        proposal, response_id, usage, proposal_fingerprint = prompt_result
        OPENAI_SHARED_PROMPT_COUNT += 1
        frame_records = []
        for midpoint_index, fraction in enumerate(fractions, start=1):
            uid = f"{pair_uid}_m{midpoint_index:02d}"
            frame_records.append({
                "uid": uid,
                "fraction": fraction,
                "output_path": image_directory / f"{uid}.png",
            })
        pair_contract = {
            "method": "sequence-cached FlowMorph interior-alpha rendering",
            "round": round_number,
            "prompt_mode": prompt_mode,
            "left_uid": left["uid"],
            "left_image_sha256": file_sha256(left["path"]),
            "left_prompt": left["prompt"],
            "right_uid": right["uid"],
            "right_image_sha256": file_sha256(right["path"]),
            "right_prompt": right["prompt"],
            "proposal_fingerprint": proposal_fingerprint,
            "shared_midpoint_prompt": proposal.prompt,
            "conditioning_mode": "piecewise_source_midpoint_target_embeddings",
            "alphas": fractions,
            "left_endpoint_fingerprint": endpoint_fingerprint(left),
            "right_endpoint_fingerprint": endpoint_fingerprint(right),
            "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
            "endpoint_optimization_steps": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
            "start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
            "render_indices": list(FLOWMORPH_RENDER_INDICES),
            "render_lora_scale": FLOWMORPH_RENDER_LORA_SCALE,
            "guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
        }
        pair_fingerprint = stable_fingerprint(pair_contract)
        completion_path = image_directory / f"{pair_uid}.flowmorph.json"
        completion = None
        completed = False
        if completion_path.is_file() and all(item["output_path"].is_file() for item in frame_records):
            completion = json.loads(completion_path.read_text(encoding="utf-8"))
            completed = completion.get("pair_fingerprint") == pair_fingerprint
        pair_jobs.append({
            "pair_uid": pair_uid,
            "left": left,
            "right": right,
            "proposal": proposal,
            "proposal_path": proposal_path,
            "response_id": response_id,
            "usage": usage,
            "frame_records": frame_records,
            "pair_contract": pair_contract,
            "pair_fingerprint": pair_fingerprint,
            "completion_path": completion_path,
            "completion": completion,
            "completed": completed,
        })

    missing_prompts = []
    for record in incoming:
        if record["prompt"] not in PROMPT_CONDITIONING_CACHE:
            missing_prompts.append(record["prompt"])
    for job in pair_jobs:
        prompt = job["proposal"].prompt
        if prompt not in PROMPT_CONDITIONING_CACHE:
            missing_prompts.append(prompt)
    missing_images = {
        record["uid"]: (
            record["path"],
            SEQUENCE_ASSET_ROOT / f"{record['uid']}.png",
        )
        for record in incoming
        if record["uid"] not in IMAGE_ASSET_CACHE
    }
    if missing_prompts or missing_images:
        new_prompts, new_images = SEQUENCE_SESSION.encode_missing_assets(
            prompts=missing_prompts,
            images=missing_images,
        )
        PROMPT_CONDITIONING_CACHE.update(new_prompts)
        IMAGE_ASSET_CACHE.update(new_images)
    print({
        "round": round_number,
        "encoded_unique_images_total": len(IMAGE_ASSET_CACHE),
        "encoded_unique_prompts_total": len(PROMPT_CONDITIONING_CACHE),
    })

    # Lazily fit endpoints and render/decode small chunks. The cache still fits
    # every unique image only once, while the first PNGs arrive after the first
    # few endpoints instead of after fitting the entire round.
    pending_jobs = []
    for job in pair_jobs:
        FLOWMORPH_PAIR_RENDER_COUNT += 1
        if job["completed"]:
            print(f"Reusing completed pair {job['pair_uid']}")
            continue
        pending_jobs.append(job)

    progress_directory = round_directory / "streaming_progress"
    progress_directory.mkdir(parents=True, exist_ok=True)
    for chunk_start in range(0, len(pending_jobs), FLOWMORPH_STREAM_PAIRS_PER_CHUNK):
        chunk_jobs = pending_jobs[
            chunk_start:chunk_start + FLOWMORPH_STREAM_PAIRS_PER_CHUNK
        ]
        chunk_frames = []
        chunk_paths = []
        fit_sequence_endpoints(
            [
                endpoint_record
                for job in chunk_jobs
                for endpoint_record in (job["left"], job["right"])
            ],
            f"Round {round_number} batched streaming fit",
        )
        for job in chunk_jobs:
            left = job["left"]
            right = job["right"]
            shared = PROMPT_CONDITIONING_CACHE[job["proposal"].prompt]
            rendered = SEQUENCE_SESSION.render_midpoints(
                source=ENDPOINT_CACHE[left["uid"]],
                target=ENDPOINT_CACHE[right["uid"]],
                source_conditioning=PROMPT_CONDITIONING_CACHE[left["prompt"]],
                target_conditioning=PROMPT_CONDITIONING_CACHE[right["prompt"]],
                midpoint_conditionings=[shared] * midpoint_count,
                alphas=fractions,
            )
            chunk_frames.extend(rendered)
            chunk_paths.extend(item["output_path"] for item in job["frame_records"])
            print(
                f"Rendered {job['pair_uid']} ({midpoint_count} interior frame(s)); "
                f"render_batch={SEQUENCE_SESSION.last_render_batch_size}; "
                "decoding with this chunk..."
            )

        SEQUENCE_SESSION.decode_frames_to_paths(chunk_frames, chunk_paths)
        print(f"Decoded with batch_size={SEQUENCE_SESSION.last_decode_batch_size}")
        for job in chunk_jobs:
            inserted = [
                {
                    "alpha": item["fraction"],
                    "image": str(item["output_path"]),
                    "shared_prompt": job["proposal"].prompt,
                    "conditioning": "piecewise source→shared midpoint→target embeddings",
                }
                for item in job["frame_records"]
            ]
            completion = {
                "status": "complete",
                "pair_uid": job["pair_uid"],
                "pair_fingerprint": job["pair_fingerprint"],
                "pair_contract": job["pair_contract"],
                "sequence_session_directory": str(SEQUENCE_SESSION_DIRECTORY),
                "left_endpoint_checkpoint_fingerprint": ENDPOINT_FINGERPRINTS[job["left"]["uid"]],
                "right_endpoint_checkpoint_fingerprint": ENDPOINT_FINGERPRINTS[job["right"]["uid"]],
                "rendered_latent_count": len(job["frame_records"]),
                "inserted": inserted,
            }
            job["completion_path"].write_text(
                json.dumps(completion, indent=2, ensure_ascii=False) + "\n",
                encoding="utf-8",
            )
            job["completion"] = completion
            job["completed"] = True

        completed_count = min(
            chunk_start + len(chunk_jobs),
            len(pending_jobs),
        )
        print({
            "round": round_number,
            "new_pairs_saved": completed_count,
            "new_pairs_total": len(pending_jobs),
            "latest_png": str(chunk_paths[-1]),
        })
        if FLOWMORPH_STREAM_DISPLAY_PROGRESS:
            representative_paths = [
                job["frame_records"][len(job["frame_records"]) // 2]["output_path"]
                for job in chunk_jobs
            ]
            representative_images = []
            for path in representative_paths:
                with Image.open(path) as opened:
                    thumbnail = opened.convert("RGB")
                    thumbnail.thumbnail((256, 256))
                    representative_images.append(thumbnail)
            chunk_number = chunk_start // FLOWMORPH_STREAM_PAIRS_PER_CHUNK + 1
            chunk_sheet_path = progress_directory / f"chunk_{chunk_number:03d}.png"
            make_contact_sheet(
                representative_images,
                chunk_sheet_path,
                columns=len(representative_images),
                labels=[job["pair_uid"] for job in chunk_jobs],
            )
            for image in representative_images:
                image.close()
            chunk_preview = Image.open(chunk_sheet_path).convert("RGB")
            chunk_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
            display(Markdown(
                f"### Round {round_number} streaming progress: "
                f"{completed_count}/{len(pending_jobs)} new pairs saved"
            ))
            display(chunk_preview)
            chunk_preview.close()
        del chunk_frames, chunk_paths
        gc.collect()

    # Completed legacy pairs may predate canonical endpoint PNGs. Restore their
    # cached fitted states and render only the missing endpoint reconstructions.
    missing_endpoint_reconstructions = [
        record
        for record in incoming
        if record["uid"] not in ENDPOINT_RECONSTRUCTION_PATHS
    ]
    if missing_endpoint_reconstructions:
        fit_sequence_endpoints(
            missing_endpoint_reconstructions,
            f"Round {round_number} endpoint reconstruction",
        )
    unresolved_endpoint_reconstructions = [
        record["uid"]
        for record in incoming
        if record["uid"] not in ENDPOINT_RECONSTRUCTION_PATHS
    ]
    if unresolved_endpoint_reconstructions:
        raise RuntimeError(
            "Canonical endpoint reconstructions are missing: "
            + ", ".join(unresolved_endpoint_reconstructions)
        )

    outgoing = []
    for job in pair_jobs:
        left = job["left"]
        right = job["right"]
        proposal = job["proposal"]
        endpoint_record = dict(left)
        endpoint_record["flowmorph_endpoint_input_path"] = str(left["path"])
        endpoint_record["flowmorph_endpoint_reconstruction_path"] = str(
            ENDPOINT_RECONSTRUCTION_PATHS[left["uid"]]
        )
        endpoint_record["canonical_endpoint_reconstruction_available"] = True
        outgoing.append(endpoint_record)
        for frame_record in job["frame_records"]:
            outgoing.append({
                "uid": frame_record["uid"],
                "kind": "flowmorph_midpoint",
                "round": round_number,
                "prompt_mode": prompt_mode,
                "fraction": frame_record["fraction"],
                "alpha": frame_record["fraction"],
                "left_uid": left["uid"],
                "right_uid": right["uid"],
                "science": proposal.science_connection,
                "visual_correspondence": proposal.visual_correspondence,
                "prompt": proposal.prompt,
                "path": str(frame_record["output_path"]),
                "proposal_path": str(job["proposal_path"]),
                "flowmorph_completion_path": str(job["completion_path"]),
                "flowmorph_sequence_session": str(SEQUENCE_SESSION_DIRECTORY),
                "flowmorph_fingerprint": job["pair_fingerprint"],
                "openai_response_id": job["response_id"],
                "usage": job["usage"],
            })

    CURRENT_RECORDS = outgoing
    round_manifest_path = round_directory / "sequence_manifest.json"
    round_manifest_path.write_text(json.dumps({
        "round": round_number,
        "cyclic": True,
        "interpolation_method": "sequence-cached true FlowMorph with canonical endpoint reconstructions",
        "endpoint_display_contract": (
            "one decoded fitted endpoint reused for incoming alpha=1 "
            "and outgoing alpha=0"
        ),
        "prompt_mode": prompt_mode,
        "one_shared_prompt_per_gap": True,
        "input_count": len(incoming),
        "midpoints_per_gap": midpoint_count,
        "alphas": fractions,
        "execution_batching": {
            "endpoint_batch_size": FLOWMORPH_ENDPOINT_BATCH_SIZE,
            "render_batch_size": FLOWMORPH_RENDER_BATCH_SIZE,
            "decode_batch_size": FLOWMORPH_DECODE_BATCH_SIZE,
            "cfg_execution_requested": FLOWMORPH_CFG_EXECUTION,
            "cfg_execution_active": SEQUENCE_SESSION.cfg_execution,
            "openai_concurrency": OPENAI_CONCURRENCY,
            "oom_backoff": FLOWMORPH_BATCH_OOM_BACKOFF,
        },
        "output_count": len(outgoing),
        "records": outgoing,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    ROUND_MANIFESTS.append(str(round_manifest_path))

    # Contact sheets use small thumbnails; never allocate a 330 × 1024px mosaic.
    round_contact_sheet = round_directory / "contact_sheet.png"
    round_images = []
    for item in outgoing:
        preview_path = item.get(
            "flowmorph_endpoint_reconstruction_path",
            item["path"],
        )
        preview_image = Image.open(preview_path).convert("RGB")
        preview_image.thumbnail((160, 160))
        round_images.append(preview_image)
    make_contact_sheet(
        round_images,
        round_contact_sheet,
        columns=min(CONTACT_SHEET_COLUMNS, len(round_images)),
        labels=[item["uid"] for item in outgoing],
    )
    for image in round_images:
        image.close()
    preview = Image.open(round_contact_sheet).convert("RGB")
    preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown(f"### Sequence FlowMorph round {round_number}: {len(outgoing)} cyclic images"))
    display(preview)
    preview.close()

FINAL_RECORDS = []
for record in CURRENT_RECORDS:
    final_record = dict(record)
    reconstruction_path = ENDPOINT_RECONSTRUCTION_PATHS.get(record["uid"])
    if reconstruction_path is not None:
        final_record["flowmorph_endpoint_input_path"] = str(record["path"])
        final_record["flowmorph_endpoint_reconstruction_path"] = str(
            reconstruction_path
        )
        final_record["path"] = str(reconstruction_path)
        final_record["canonical_endpoint_reconstruction_used"] = True
    else:
        final_record["canonical_endpoint_reconstruction_used"] = False
    FINAL_RECORDS.append(final_record)

FINAL_SEQUENCE_MANIFEST = RUN_DIRECTORY / "metadata" / "one_round_flowmorph_conditioning_sequence.json"
FINAL_SEQUENCE_MANIFEST.write_text(json.dumps({
    "project": PROJECT_NAME,
    "cyclic": True,
    "interpolation_method": (
        "one-model sequence-cached true FlowMorph with one canonical "
        "reconstruction per fitted endpoint"
    ),
    "endpoint_display_contract": (
        "the same decoded fitted endpoint file is reused for incoming alpha=1 "
        "and outgoing alpha=0"
    ),
    "anchor_count": len(BASE_RECORDS),
    "round_specs": FLOWMORPH_ROUND_SPECS,
    "model_loads": 1,
    "backward_probes": 1,
    "unique_endpoint_fits": UNIQUE_ENDPOINT_FIT_COUNT,
    "canonical_endpoint_reconstruction_count": len(
        ENDPOINT_RECONSTRUCTION_PATHS
    ),
    "pair_renders": FLOWMORPH_PAIR_RENDER_COUNT,
    "one_openai_prompt_per_gap": True,
    "openai_prompt_count": OPENAI_SHARED_PROMPT_COUNT,
    "execution_batching": {
        "endpoint_batch_size": FLOWMORPH_ENDPOINT_BATCH_SIZE,
        "render_batch_size": FLOWMORPH_RENDER_BATCH_SIZE,
        "decode_batch_size": FLOWMORPH_DECODE_BATCH_SIZE,
        "cfg_execution_requested": FLOWMORPH_CFG_EXECUTION,
        "cfg_execution_active": SEQUENCE_SESSION.cfg_execution,
        "openai_concurrency": OPENAI_CONCURRENCY,
        "oom_backoff": FLOWMORPH_BATCH_OOM_BACKOFF,
    },
    "final_count": len(FINAL_RECORDS),
    "round_manifests": ROUND_MANIFESTS,
    "records": FINAL_RECORDS,
}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print({
    "final_images": len(FINAL_RECORDS),
    "unique_endpoint_fits": UNIQUE_ENDPOINT_FIT_COUNT,
    "canonical_endpoint_reconstructions": len(
        ENDPOINT_RECONSTRUCTION_PATHS
    ),
    "pair_renders": FLOWMORPH_PAIR_RENDER_COUNT,
    "model_loads": 1,
    "backward_probes": 1,
    "manifest": str(FINAL_SEQUENCE_MANIFEST),
})


# Freeze the exact per-gap conditioning series before releasing FlowMorph.
# Every series has: canonical left endpoint, 12 interiors, canonical right.
LTX_GAP_RECORDS = []
for job in pair_jobs:
    condition_paths = [
        str(ENDPOINT_RECONSTRUCTION_PATHS[job["left"]["uid"]]),
        *[
            str(frame_record["output_path"])
            for frame_record in job["frame_records"]
        ],
        str(ENDPOINT_RECONSTRUCTION_PATHS[job["right"]["uid"]]),
    ]
    if len(condition_paths) != 14:
        raise RuntimeError(
            f"{job['pair_uid']} has {len(condition_paths)} conditions; expected 14"
        )
    missing = [path for path in condition_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            f"{job['pair_uid']} has missing conditioning images: {missing}"
        )
    LTX_GAP_RECORDS.append({
        "pair_uid": job["pair_uid"],
        "left_uid": job["left"]["uid"],
        "right_uid": job["right"]["uid"],
        "source_science": job["left"]["science"],
        "target_science": job["right"]["science"],
        "science_connection": job["proposal"].science_connection,
        "shared_midpoint_prompt": job["proposal"].prompt,
        "condition_paths": condition_paths,
    })

LTX_CONDITIONING_MANIFEST = (
    RUN_DIRECTORY / "metadata" / "ltx_conditioning_gaps.json"
)
LTX_CONDITIONING_MANIFEST.write_text(json.dumps({
    "cyclic": True,
    "gap_count": len(LTX_GAP_RECORDS),
    "flowmorph_interiors_per_gap": 12,
    "conditions_per_gap": 14,
    "records": LTX_GAP_RECORDS,
}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print({
    "ltx_conditioning_manifest": str(LTX_CONDITIONING_MANIFEST),
    "cyclic_gaps": len(LTX_GAP_RECORDS),
    "conditions_per_gap": 14,
})


## 11. Explicitly release FLUX/FlowMorph before loading LTX

LTX is a separate 13B video stack. This cell moves every known FLUX
component to CPU, removes references and hooks, runs garbage
collection, empties PyTorch's CUDA allocator, and reports remaining
memory. A high residual is reported clearly before the 13B load.


In [ ]:
import gc
import shutil
import subprocess
import sys
import torch

sequence_runner_for_release = globals().get("SEQUENCE_RUNNER")
flowmorph_pipeline = getattr(
    sequence_runner_for_release,
    "pipeline",
    None,
)
if flowmorph_pipeline is not None:
    maybe_free = getattr(flowmorph_pipeline, "maybe_free_model_hooks", None)
    if callable(maybe_free):
        maybe_free()
    for component_name in ("transformer", "vae", "text_encoder"):
        component = getattr(flowmorph_pipeline, component_name, None)
        if component is not None and callable(getattr(component, "to", None)):
            component.to("cpu")

for variable_name in (
    "ENDPOINT_CACHE",
    "IMAGE_ASSET_CACHE",
    "PROMPT_CONDITIONING_CACHE",
    "ENDPOINT_RECONSTRUCTION_PATHS",
    "SEQUENCE_SESSION",
    "SEQUENCE_RUNNER",
    "session_config",
    "session_template",
    "FLUX_PROMPT_TOKENIZER",
    "flowmorph_pipeline",
    "sequence_runner_for_release",
):
    value = globals().pop(variable_name, None)
    if value is not None:
        del value

gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except RuntimeError:
        pass
    torch.cuda.reset_peak_memory_stats()
    LTX_PRELOAD_ALLOCATED_GIB = torch.cuda.memory_allocated() / 1024**3
    LTX_PRELOAD_RESERVED_GIB = torch.cuda.memory_reserved() / 1024**3
    LTX_GPU_TOTAL_GIB = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )
else:
    raise RuntimeError("LTX-Video 13B requires a CUDA runtime")

# The FLUX repository is about 32 GiB and the LTX 13B repository is
# about 45 GiB. They need not coexist after all FlowMorph PNGs and
# endpoint checkpoints have been saved. Remove only the exact,
# reproducible local FLUX model cache—not Drive outputs or LoRA files.
flux_cache_directory = (
    Path(HF_CACHE_DIR)
    / ("models--" + MODEL_ID.replace("/", "--"))
)
if DELETE_LOCAL_FLUX_CACHE_BEFORE_LTX and flux_cache_directory.is_dir():
    expected_parent = Path(HF_CACHE_DIR).resolve()
    if flux_cache_directory.resolve().parent != expected_parent:
        raise RuntimeError(
            "Refusing to delete an unexpected FLUX cache path: "
            f"{flux_cache_directory}"
        )
    shutil.rmtree(flux_cache_directory)
    print(
        "Removed the released local FLUX model cache to make room "
        f"for LTX: {flux_cache_directory}"
    )

local_flowmorph_work = (
    Path(LOCAL_ASSET_ROOT)
    / PROJECT_NAME
    / "sequence_work"
)
if (
    DELETE_LOCAL_FLOWMORPH_WORK_BEFORE_LTX
    and local_flowmorph_work.is_dir()
):
    expected_work_parent = (
        Path(LOCAL_ASSET_ROOT) / PROJECT_NAME
    ).resolve()
    if local_flowmorph_work.resolve().parent != expected_work_parent:
        raise RuntimeError(
            "Refusing to delete an unexpected FlowMorph work path: "
            f"{local_flowmorph_work}"
        )
    shutil.rmtree(local_flowmorph_work)
    print(
        "Removed completed local FlowMorph scratch work: "
        f"{local_flowmorph_work}"
    )

if CLEAN_PIP_CACHE_BEFORE_LTX:
    pip_cleanup = subprocess.run(
        [sys.executable, "-m", "pip", "cache", "purge"],
        capture_output=True,
        text=True,
    )
    print(
        "pip cache cleanup:",
        (
            pip_cleanup.stdout.strip()
            or pip_cleanup.stderr.strip()
            or f"exit {pip_cleanup.returncode}"
        ),
    )

Path(LTX_CACHE_DIR).mkdir(parents=True, exist_ok=True)
LTX_DISK_FREE_GIB = (
    shutil.disk_usage(LTX_CACHE_DIR).free / 1024**3
)
print({
    "gpu": torch.cuda.get_device_name(0),
    "total_gib": round(LTX_GPU_TOTAL_GIB, 3),
    "allocated_gib_after_flux_release": round(
        LTX_PRELOAD_ALLOCATED_GIB, 3
    ),
    "reserved_gib_after_flux_release": round(
        LTX_PRELOAD_RESERVED_GIB, 3
    ),
    "ltx_cache_directory": str(Path(LTX_CACHE_DIR)),
    "local_disk_free_gib": round(LTX_DISK_FREE_GIB, 3),
})
if LTX_PRELOAD_RESERVED_GIB > LTX_MAX_RESERVED_GIB_BEFORE_LOAD:
    print(
        "WARNING: CUDA still reserves more than the configured "
        "preload threshold. If LTX OOMs, restart the runtime, set "
        "RESUME_RUN_DIRECTORY to this run, and rerun through here."
    )


## 11a. Read-only local-disk investigation

This cell does not delete anything. It inventories Colab's local
filesystem while explicitly excluding the mounted Google Drive tree.
It reports the largest local directories/files, individual Hugging
Face repository caches, interrupted downloads, pip/Xet caches, and
FlowMorph scratch space, then saves the same JSON report to Drive.


In [ ]:
import os
import shutil
from pathlib import Path

def local_tree_size_bytes(path):
    path = Path(path)
    if not path.exists() or path.is_symlink():
        return 0
    if path.is_file():
        try:
            return path.stat().st_size
        except OSError:
            return 0
    total = 0
    for root, directory_names, file_names in os.walk(
        path,
        followlinks=False,
    ):
        root_path = Path(root)
        directory_names[:] = [
            name
            for name in directory_names
            if not (root_path / name).is_symlink()
        ]
        for name in file_names:
            file_path = root_path / name
            if file_path.is_symlink():
                continue
            try:
                total += file_path.stat().st_size
            except OSError:
                pass
    return total

def gibibytes(byte_count):
    return round(byte_count / 1024**3, 3)

disk = shutil.disk_usage("/")
content_root = Path("/content")
local_entries = []
if content_root.is_dir():
    for child in content_root.iterdir():
        # Drive is persistent remote storage and is deliberately
        # excluded from the local ephemeral-disk diagnosis.
        if child.name == "drive":
            continue
        local_entries.append({
            "path": str(child),
            "bytes": local_tree_size_bytes(child),
        })
for root in (Path("/root/.cache"), Path("/tmp")):
    if root.is_dir():
        for child in root.iterdir():
            local_entries.append({
                "path": str(child),
                "bytes": local_tree_size_bytes(child),
            })
local_entries.sort(key=lambda item: item["bytes"], reverse=True)

cache_root = Path(LTX_CACHE_DIR)
huggingface_repositories = []
if cache_root.is_dir():
    for child in cache_root.iterdir():
        if child.is_dir() and child.name.startswith("models--"):
            huggingface_repositories.append({
                "path": str(child),
                "bytes": local_tree_size_bytes(child),
            })
huggingface_repositories.sort(
    key=lambda item: item["bytes"],
    reverse=True,
)

incomplete_files = []
for search_root in {
    Path(HF_CACHE_DIR),
    Path(LTX_CACHE_DIR),
    Path("/root/.cache/huggingface"),
}:
    if not search_root.is_dir():
        continue
    for path in search_root.rglob("*.incomplete"):
        if path.is_file() and not path.is_symlink():
            incomplete_files.append({
                "path": str(path),
                "bytes": path.stat().st_size,
            })
incomplete_files.sort(
    key=lambda item: item["bytes"],
    reverse=True,
)

largest_files = []
largest_file_roots = {
    Path(HF_CACHE_DIR),
    Path(LTX_CACHE_DIR),
    Path(LOCAL_ASSET_ROOT),
    Path("/root/.cache"),
}
for search_root in largest_file_roots:
    if not search_root.is_dir():
        continue
    for root, directory_names, file_names in os.walk(
        search_root,
        followlinks=False,
    ):
        root_path = Path(root)
        directory_names[:] = [
            name
            for name in directory_names
            if not (root_path / name).is_symlink()
        ]
        for name in file_names:
            path = root_path / name
            if path.is_symlink():
                continue
            try:
                largest_files.append({
                    "path": str(path),
                    "bytes": path.stat().st_size,
                })
            except OSError:
                pass
largest_files.sort(
    key=lambda item: item["bytes"],
    reverse=True,
)

DISK_INVESTIGATION_REPORT = {
    "read_only": True,
    "google_drive_excluded": True,
    "filesystem": {
        "total_gib": gibibytes(disk.total),
        "used_gib": gibibytes(disk.used),
        "free_gib": gibibytes(disk.free),
    },
    "configured_paths": {
        "hf_cache": HF_CACHE_DIR,
        "ltx_cache": LTX_CACHE_DIR,
        "local_asset_root": LOCAL_ASSET_ROOT,
    },
    "largest_local_entries": [
        {
            "path": item["path"],
            "gib": gibibytes(item["bytes"]),
        }
        for item in local_entries[:30]
    ],
    "huggingface_repository_caches": [
        {
            "path": item["path"],
            "gib": gibibytes(item["bytes"]),
        }
        for item in huggingface_repositories
    ],
    "incomplete_downloads": [
        {
            "path": item["path"],
            "gib": gibibytes(item["bytes"]),
        }
        for item in incomplete_files
    ],
    "largest_local_cache_files": [
        {
            "path": item["path"],
            "gib": gibibytes(item["bytes"]),
        }
        for item in largest_files[:30]
    ],
}
DISK_INVESTIGATION_PATH = (
    RUN_DIRECTORY
    / "metadata"
    / "disk_investigation.json"
)
DISK_INVESTIGATION_PATH.write_text(
    json.dumps(
        DISK_INVESTIGATION_REPORT,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

print("Filesystem:", DISK_INVESTIGATION_REPORT["filesystem"])
print("\nLargest local entries:")
for item in DISK_INVESTIGATION_REPORT["largest_local_entries"]:
    print(f"  {item['gib']:8.3f} GiB  {item['path']}")
print("\nHugging Face repository caches:")
for item in DISK_INVESTIGATION_REPORT[
    "huggingface_repository_caches"
]:
    print(f"  {item['gib']:8.3f} GiB  {item['path']}")
print("\nIncomplete downloads:")
if not DISK_INVESTIGATION_REPORT["incomplete_downloads"]:
    print("  none")
for item in DISK_INVESTIGATION_REPORT["incomplete_downloads"]:
    print(f"  {item['gib']:8.3f} GiB  {item['path']}")
print("\nLargest local cache files:")
for item in DISK_INVESTIGATION_REPORT[
    "largest_local_cache_files"
]:
    print(f"  {item['gib']:8.3f} GiB  {item['path']}")
print("\nSaved report:", DISK_INVESTIGATION_PATH)


## 12. Validate and fingerprint the LTX conditioning jobs

Fourteen stills are assigned to frames `0, 8, …, 104`, so each clip
has 105 frames. The terminal frame is conditioned on the next anchor.
When clips are assembled, that one terminal frame is omitted from each
segment; the next clip begins on the identical canonical endpoint, and
the final segment approaches the first anchor before playback loops.


In [ ]:
from pathlib import Path

LTX_ROOT = RUN_DIRECTORY / "ltx13b"
LTX_CLIP_DIRECTORY = LTX_ROOT / "clips"
LTX_METADATA_DIRECTORY = LTX_ROOT / "metadata"
LTX_CLIP_DIRECTORY.mkdir(parents=True, exist_ok=True)
LTX_METADATA_DIRECTORY.mkdir(parents=True, exist_ok=True)

def make_ltx_motion_prompt(record):
    prompt = LTX_MOTION_PROMPT_TEMPLATE.format(
        source_science=record["source_science"],
        target_science=record["target_science"],
        science_connection=record["science_connection"],
    ).strip()
    word_count = len(prompt.split())
    if word_count > LTX_MAX_PROMPT_WORDS:
        raise ValueError(
            f"{record['pair_uid']} LTX prompt has {word_count} words; "
            f"maximum is {LTX_MAX_PROMPT_WORDS}"
        )
    return prompt, word_count

LTX_FRAME_INDICES = [
    index * LTX_FRAMES_PER_CONDITION_INTERVAL
    for index in range(14)
]
LTX_NUM_FRAMES = LTX_FRAME_INDICES[-1] + 1
if LTX_NUM_FRAMES % 8 != 1:
    raise RuntimeError("LTX frame count must be N*8+1")

LTX_JOBS = []
for gap_index, record in enumerate(LTX_GAP_RECORDS):
    prompt, prompt_word_count = make_ltx_motion_prompt(record)
    clip_path = (
        LTX_CLIP_DIRECTORY
        / f"{gap_index:04d}_{record['pair_uid']}.mp4"
    )
    metadata_path = (
        LTX_METADATA_DIRECTORY
        / f"{gap_index:04d}_{record['pair_uid']}.json"
    )
    contract = {
        "model_id": LTX_MODEL_ID,
        "model_revision": LTX_MODEL_REVISION,
        "upsampler_id": (
            LTX_UPSAMPLER_ID
            if LTX_USE_TWO_STAGE_UPSCALING
            else None
        ),
        "upsampler_revision": (
            LTX_UPSAMPLER_REVISION
            if LTX_USE_TWO_STAGE_UPSCALING
            else None
        ),
        "pair_uid": record["pair_uid"],
        "condition_sha256": [
            file_sha256(path)
            for path in record["condition_paths"]
        ],
        "condition_frame_indices": LTX_FRAME_INDICES,
        "conditioning_strength": LTX_CONDITIONING_STRENGTH,
        "num_frames": LTX_NUM_FRAMES,
        "fps": LTX_FPS,
        "final_size": [LTX_FINAL_WIDTH, LTX_FINAL_HEIGHT],
        "prompt": prompt,
        "negative_prompt": LTX_NEGATIVE_PROMPT,
        "first_pass_timesteps": LTX_FIRST_PASS_TIMESTEPS,
        "second_pass_timesteps": LTX_SECOND_PASS_TIMESTEPS,
        "image_cond_noise_scale": LTX_IMAGE_COND_NOISE_SCALE,
        "decode_timestep": LTX_DECODE_TIMESTEP,
        "decode_noise_scale": LTX_DECODE_NOISE_SCALE,
        "tone_map_compression_ratio": (
            LTX_TONE_MAP_COMPRESSION_RATIO
        ),
    }
    fingerprint = stable_fingerprint(contract)
    reusable = False
    if (
        REUSE_EXISTING_LTX_CLIPS
        and clip_path.is_file()
        and metadata_path.is_file()
    ):
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        reusable = (
            saved.get("status") == "complete"
            and saved.get("fingerprint") == fingerprint
        )
    LTX_JOBS.append({
        "gap_index": gap_index,
        "record": record,
        "prompt": prompt,
        "prompt_word_count": prompt_word_count,
        "clip_path": clip_path,
        "metadata_path": metadata_path,
        "contract": contract,
        "fingerprint": fingerprint,
        "reusable": reusable,
    })

LTX_PENDING_JOBS = [job for job in LTX_JOBS if not job["reusable"]]
print({
    "cyclic_gap_clips": len(LTX_JOBS),
    "pending_clips": len(LTX_PENDING_JOBS),
    "conditions_per_clip": 14,
    "condition_frame_indices": LTX_FRAME_INDICES,
    "frames_per_clip": LTX_NUM_FRAMES,
    "seconds_per_clip": round(LTX_NUM_FRAMES / LTX_FPS, 3),
    "expected_loop_seconds_without_duplicate_endpoints": round(
        len(LTX_JOBS) * (LTX_NUM_FRAMES - 1) / LTX_FPS,
        3,
    ),
})


## 13. Load one memory-managed LTX 0.9.8 13B stack

The 13B distilled transformer is stored layerwise in FP8 and computed
in BF16. Group offload transfers only the layers currently needed to
CUDA. The pipeline is retained across all pending gaps; if every clip
fingerprint already matches, no 13B model is loaded.


In [ ]:
# Make this cell safe to rerun after an interrupted download or a
# failed partial model construction.
for stale_name in (
    "LTX_PIPE",
    "LTX_UPSCALE_PIPE",
    "LTX_UPSAMPLER",
    "ltx_transformer",
):
    stale = globals().pop(stale_name, None)
    if stale is not None:
        try:
            stale.to("cpu")
        except (AttributeError, RuntimeError, ValueError):
            pass
        del stale
gc.collect()
torch.cuda.empty_cache()

LTX_PIPE = None
LTX_UPSCALE_PIPE = None
LTX_UPSAMPLER = None

if LTX_PENDING_JOBS:
    import os
    import huggingface_hub.constants as hf_hub_constants
    from huggingface_hub import snapshot_download
    from huggingface_hub.constants import HF_XET_CACHE
    from diffusers import AutoModel, LTXConditionPipeline
    from diffusers.hooks import apply_group_offloading
    from diffusers.pipelines.ltx.modeling_latent_upsampler import (
        LTXLatentUpsamplerModel,
    )
    from diffusers.pipelines.ltx.pipeline_ltx_condition import (
        LTXVideoCondition,
    )
    from diffusers import LTXLatentUpsamplePipeline

    # Xet reconstructs a target file while retaining a separate
    # chunk cache, which can temporarily duplicate a large shard.
    # Plain resumable HTTP is slower but materially safer on
    # Colab's constrained ephemeral disk.
    if LTX_DISABLE_XET_FOR_DISK_SAFETY:
        os.environ["HF_HUB_DISABLE_XET"] = "1"
        hf_hub_constants.HF_HUB_DISABLE_XET = True

    # Ask the Hub what is still missing before starting another
    # multi-gigabyte Xet transfer. This accounts for completed files
    # from a prior partial attempt and fails with an actionable disk
    # message rather than an opaque reconstruction error.
    ltx_download_plan = snapshot_download(
        repo_id=LTX_MODEL_ID,
        revision=LTX_MODEL_REVISION,
        cache_dir=LTX_CACHE_DIR,
        allow_patterns=[
            "model_index.json",
            "scheduler/*",
            "tokenizer/*",
            "text_encoder/*",
            "transformer/*",
            "vae/*",
        ],
        dry_run=True,
    )
    ltx_missing_bytes = sum(
        item.file_size
        for item in ltx_download_plan
        if item.will_download
    )
    if LTX_USE_TWO_STAGE_UPSCALING:
        upsampler_download_plan = snapshot_download(
            repo_id=LTX_UPSAMPLER_ID,
            revision=LTX_UPSAMPLER_REVISION,
            cache_dir=LTX_CACHE_DIR,
            dry_run=True,
        )
        ltx_missing_bytes += sum(
            item.file_size
            for item in upsampler_download_plan
            if item.will_download
        )
    ltx_free_bytes = shutil.disk_usage(LTX_CACHE_DIR).free
    ltx_required_bytes = (
        ltx_missing_bytes
        + int(LTX_DOWNLOAD_HEADROOM_GIB * 1024**3)
    )
    print({
        "ltx_download_remaining_gib": round(
            ltx_missing_bytes / 1024**3, 3
        ),
        "disk_free_gib": round(ltx_free_bytes / 1024**3, 3),
        "required_including_headroom_gib": round(
            ltx_required_bytes / 1024**3, 3
        ),
    })
    if (
        CLEAN_INTERRUPTED_LTX_DOWNLOADS_IF_NEEDED
        and (
            ltx_free_bytes < ltx_required_bytes
            or LTX_DISABLE_XET_FOR_DISK_SAFETY
        )
    ):
        # Hugging Face checks for a shard's full target size even
        # when a previous `.incomplete` reconstruction exists.
        # Xet also retains a separate temporary chunk cache. Keep
        # every completed blob, but remove these reproducible
        # interrupted-transfer artifacts so the retry can proceed.
        incomplete_paths = []
        for repo_id in (
            LTX_MODEL_ID,
            LTX_UPSAMPLER_ID,
        ):
            repo_cache = (
                Path(LTX_CACHE_DIR)
                / ("models--" + repo_id.replace("/", "--"))
            )
            if repo_cache.is_dir():
                incomplete_paths.extend(
                    path
                    for path in repo_cache.rglob("*.incomplete")
                    if path.is_file()
                )
        incomplete_bytes = sum(
            path.stat().st_size
            for path in incomplete_paths
        )
        for path in incomplete_paths:
            path.unlink()

        xet_cache_path = Path(HF_XET_CACHE)
        xet_cache_bytes = 0
        if xet_cache_path.is_dir():
            xet_cache_bytes = sum(
                path.stat().st_size
                for path in xet_cache_path.rglob("*")
                if path.is_file()
            )
            shutil.rmtree(xet_cache_path)

        ltx_free_bytes = shutil.disk_usage(LTX_CACHE_DIR).free
        print({
            "interrupted_ltx_files_removed": len(
                incomplete_paths
            ),
            "incomplete_ltx_gib_reclaimed": round(
                incomplete_bytes / 1024**3, 3
            ),
            "xet_temporary_gib_reclaimed": round(
                xet_cache_bytes / 1024**3, 3
            ),
            "disk_free_after_transfer_cleanup_gib": round(
                ltx_free_bytes / 1024**3, 3
            ),
        })
    if ltx_free_bytes < ltx_required_bytes:
        raise RuntimeError(
            "Not enough local disk for the remaining LTX download. "
            "Delete unused /content caches or set LTX_CACHE_DIR to "
            "a Google Drive directory with at least "
            f"{ltx_required_bytes / 1024**3:.1f} GiB free."
        )

    ltx_transformer = AutoModel.from_pretrained(
        LTX_MODEL_ID,
        subfolder="transformer",
        revision=LTX_MODEL_REVISION,
        cache_dir=LTX_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    if LTX_ENABLE_FP8_LAYERWISE_STORAGE:
        ltx_transformer.enable_layerwise_casting(
            storage_dtype=torch.float8_e4m3fn,
            compute_dtype=torch.bfloat16,
        )

    LTX_PIPE = LTXConditionPipeline.from_pretrained(
        LTX_MODEL_ID,
        revision=LTX_MODEL_REVISION,
        cache_dir=LTX_CACHE_DIR,
        transformer=ltx_transformer,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    LTX_PIPE.vae.enable_tiling()
    LTX_PIPE.vae.enable_slicing()

    cuda_device = torch.device("cuda")
    cpu_device = torch.device("cpu")
    if LTX_ENABLE_GROUP_OFFLOAD:
        LTX_PIPE.transformer.enable_group_offload(
            onload_device=cuda_device,
            offload_device=cpu_device,
            offload_type="leaf_level",
            use_stream=LTX_GROUP_OFFLOAD_USE_STREAM,
            low_cpu_mem_usage=LTX_GROUP_OFFLOAD_LOW_CPU_MEMORY,
        )
        apply_group_offloading(
            LTX_PIPE.text_encoder,
            onload_device=cuda_device,
            offload_device=cpu_device,
            offload_type="block_level",
            num_blocks_per_group=2,
            use_stream=LTX_GROUP_OFFLOAD_USE_STREAM,
            low_cpu_mem_usage=LTX_GROUP_OFFLOAD_LOW_CPU_MEMORY,
        )
        apply_group_offloading(
            LTX_PIPE.vae,
            onload_device=cuda_device,
            offload_device=cpu_device,
            offload_type="leaf_level",
            use_stream=False,
        )
    else:
        LTX_PIPE.enable_model_cpu_offload()

    if LTX_USE_TWO_STAGE_UPSCALING:
        LTX_UPSAMPLER = LTXLatentUpsamplerModel.from_pretrained(
            LTX_UPSAMPLER_ID,
            revision=LTX_UPSAMPLER_REVISION,
            cache_dir=LTX_CACHE_DIR,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
        )
        LTX_UPSCALE_PIPE = LTXLatentUpsamplePipeline(
            vae=LTX_PIPE.vae,
            latent_upsampler=LTX_UPSAMPLER,
        )

    print({
        "ltx_model": LTX_MODEL_ID,
        "model_loads": 1,
        "fp8_layerwise_storage": (
            LTX_ENABLE_FP8_LAYERWISE_STORAGE
        ),
        "group_offload": LTX_ENABLE_GROUP_OFFLOAD,
        "two_stage_upscaling": LTX_USE_TWO_STAGE_UPSCALING,
    })
else:
    print("Every LTX clip fingerprint matches; model load skipped.")


## 14. Render and save each conditioned LTX gap

Results are written immediately. The low-resolution distilled pass,
optional latent upscale, and short texture-refinement pass all reuse
the same 14 conditions. Each saved segment omits its final exact
endpoint to avoid a duplicated frame when clips are concatenated.


In [ ]:
import subprocess
import imageio.v2 as imageio
import imageio_ffmpeg
import numpy as np
from IPython.display import Video

def round_down_for_ltx(value, divisor):
    rounded = value - value % divisor
    if rounded < divisor:
        raise ValueError(f"Cannot round {value} to a positive LTX size")
    return rounded

def render_ltx_job(job):
    record = job["record"]
    condition_images = []
    try:
        for path in record["condition_paths"]:
            with Image.open(path) as opened:
                condition_images.append(opened.convert("RGB"))
        conditions = [
            LTXVideoCondition(
                image=image,
                frame_index=frame_index,
                strength=LTX_CONDITIONING_STRENGTH,
            )
            for image, frame_index in zip(
                condition_images,
                LTX_FRAME_INDICES,
                strict=True,
            )
        ]
        generator = torch.Generator(device="cuda").manual_seed(
            BASE_SEED + 100_000 + job["gap_index"]
        )
        compression = LTX_PIPE.vae_spatial_compression_ratio
        downscale = 2 / 3 if LTX_USE_TWO_STAGE_UPSCALING else 1.0
        first_height = round_down_for_ltx(
            int(LTX_FINAL_HEIGHT * downscale),
            compression,
        )
        first_width = round_down_for_ltx(
            int(LTX_FINAL_WIDTH * downscale),
            compression,
        )
        first_result = LTX_PIPE(
            conditions=conditions,
            prompt=job["prompt"],
            negative_prompt=LTX_NEGATIVE_PROMPT,
            width=first_width,
            height=first_height,
            num_frames=LTX_NUM_FRAMES,
            frame_rate=LTX_FPS,
            timesteps=LTX_FIRST_PASS_TIMESTEPS,
            decode_timestep=LTX_DECODE_TIMESTEP,
            decode_noise_scale=LTX_DECODE_NOISE_SCALE,
            image_cond_noise_scale=LTX_IMAGE_COND_NOISE_SCALE,
            guidance_scale=LTX_GUIDANCE_SCALE,
            guidance_rescale=LTX_GUIDANCE_RESCALE,
            generator=generator,
            output_type=(
                "latent"
                if LTX_USE_TWO_STAGE_UPSCALING
                else "pil"
            ),
        )

        if LTX_USE_TWO_STAGE_UPSCALING:
            first_latents = first_result.frames
            del first_result
            LTX_UPSAMPLER.to("cuda")
            upscaled_latents = LTX_UPSCALE_PIPE(
                latents=first_latents,
                adain_factor=LTX_ADAIN_FACTOR,
                tone_map_compression_ratio=(
                    LTX_TONE_MAP_COMPRESSION_RATIO
                ),
                output_type="latent",
            ).frames
            LTX_UPSAMPLER.to("cpu")
            del first_latents
            gc.collect()
            torch.cuda.empty_cache()

            render_height = first_height * 2
            render_width = first_width * 2
            frames = LTX_PIPE(
                conditions=conditions,
                prompt=job["prompt"],
                negative_prompt=LTX_NEGATIVE_PROMPT,
                width=render_width,
                height=render_height,
                num_frames=LTX_NUM_FRAMES,
                denoise_strength=LTX_UPSCALE_DENOISE_STRENGTH,
                timesteps=LTX_SECOND_PASS_TIMESTEPS,
                latents=upscaled_latents,
                decode_timestep=LTX_DECODE_TIMESTEP,
                decode_noise_scale=LTX_DECODE_NOISE_SCALE,
                image_cond_noise_scale=LTX_IMAGE_COND_NOISE_SCALE,
                guidance_scale=LTX_GUIDANCE_SCALE,
                guidance_rescale=LTX_GUIDANCE_RESCALE,
                generator=generator,
                output_type="pil",
            ).frames[0]
            del upscaled_latents
        else:
            frames = first_result.frames[0]
            del first_result

        frames = [
            frame.resize(
                (LTX_FINAL_WIDTH, LTX_FINAL_HEIGHT),
                Image.Resampling.LANCZOS,
            )
            for frame in frames
        ]
        # Drop the terminal exact endpoint. The next clip starts on
        # that same endpoint; the last clip approaches frame zero.
        with imageio.get_writer(
            str(job["clip_path"]),
            fps=LTX_FPS,
            codec="libx264",
            quality=None,
            macro_block_size=None,
            output_params=[
                "-crf",
                str(LTX_VIDEO_CRF),
                "-pix_fmt",
                "yuv420p",
                "-movflags",
                "+faststart",
            ],
        ) as writer:
            for frame in frames[:-1]:
                writer.append_data(np.asarray(frame))
        job["metadata_path"].write_text(json.dumps({
            "status": "complete",
            "fingerprint": job["fingerprint"],
            "contract": job["contract"],
            "prompt_word_count": job["prompt_word_count"],
            "saved_frames": len(frames) - 1,
            "omitted_terminal_condition_frame": True,
            "clip_path": str(job["clip_path"]),
        }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        saved_frame_count = len(frames) - 1
        for frame in frames:
            frame.close()
        return saved_frame_count
    finally:
        for image in condition_images:
            image.close()

for pending_index, job in enumerate(LTX_PENDING_JOBS, start=1):
    print(
        f"LTX gap {pending_index}/{len(LTX_PENDING_JOBS)}: "
        f"{job['record']['left_uid']} → {job['record']['right_uid']}"
    )
    saved_frames = render_ltx_job(job)
    print({
        "saved": str(job["clip_path"]),
        "frames": saved_frames,
        "prompt_words": job["prompt_word_count"],
    })
    if LTX_DISPLAY_EACH_CLIP:
        display(Video(
            filename=str(job["clip_path"]),
            embed=False,
            width=LTX_DISPLAY_WIDTH,
        ))
    gc.collect()
    torch.cuda.empty_cache()

missing_clips = [
    str(job["clip_path"])
    for job in LTX_JOBS
    if not job["clip_path"].is_file()
]
if missing_clips:
    raise FileNotFoundError(
        "LTX rendering ended with missing clips: "
        + ", ".join(missing_clips)
    )


## 15. Concatenate the cyclic clips, audit, preview, and release LTX

All per-gap MP4s share the same encoding settings, so FFmpeg can join
them without another lossy video encode. The manifest records the
FlowMorph conditioning contract, every clip fingerprint, frame count,
model/memory settings, and the final circular duration.


In [ ]:
concat_path = LTX_ROOT / "concat.txt"

def ffmpeg_concat_line(path):
    absolute = str(Path(path).resolve())
    return "file '" + absolute.replace("'", "'\\''") + "'"

concat_path.write_text(
    "\n".join(
        ffmpeg_concat_line(job["clip_path"])
        for job in LTX_JOBS
    ) + "\n",
    encoding="utf-8",
)
LTX_FINAL_VIDEO_PATH = (
    LTX_ROOT / "flowmorph_conditioned_ltx13b_cyclic.mp4"
)
ffmpeg_executable = imageio_ffmpeg.get_ffmpeg_exe()
subprocess.check_call([
    ffmpeg_executable,
    "-y",
    "-f",
    "concat",
    "-safe",
    "0",
    "-i",
    str(concat_path),
    "-c",
    "copy",
    "-movflags",
    "+faststart",
    str(LTX_FINAL_VIDEO_PATH),
])

LTX_FINAL_MANIFEST = LTX_ROOT / "ltx13b_video_manifest.json"
LTX_FINAL_MANIFEST.write_text(json.dumps({
    "project": PROJECT_NAME,
    "cyclic": True,
    "model_id": LTX_MODEL_ID,
    "model_revision": LTX_MODEL_REVISION,
    "upsampler_id": (
        LTX_UPSAMPLER_ID
        if LTX_USE_TWO_STAGE_UPSCALING
        else None
    ),
    "upsampler_revision": (
        LTX_UPSAMPLER_REVISION
        if LTX_USE_TWO_STAGE_UPSCALING
        else None
    ),
    "flowmorph_rounds": 1,
    "flowmorph_interiors_per_gap": 12,
    "conditions_per_gap": 14,
    "condition_frame_indices": LTX_FRAME_INDICES,
    "frames_generated_per_gap": LTX_NUM_FRAMES,
    "frames_saved_per_gap": LTX_NUM_FRAMES - 1,
    "gap_count": len(LTX_JOBS),
    "final_frame_count": len(LTX_JOBS) * (LTX_NUM_FRAMES - 1),
    "fps": LTX_FPS,
    "duration_seconds": (
        len(LTX_JOBS) * (LTX_NUM_FRAMES - 1) / LTX_FPS
    ),
    "final_video": str(LTX_FINAL_VIDEO_PATH),
    "flowmorph_conditioning_manifest": str(
        LTX_CONDITIONING_MANIFEST
    ),
    "fp8_layerwise_storage": LTX_ENABLE_FP8_LAYERWISE_STORAGE,
    "group_offload": LTX_ENABLE_GROUP_OFFLOAD,
    "clips": [
        {
            "pair_uid": job["record"]["pair_uid"],
            "left_uid": job["record"]["left_uid"],
            "right_uid": job["record"]["right_uid"],
            "path": str(job["clip_path"]),
            "fingerprint": job["fingerprint"],
        }
        for job in LTX_JOBS
    ],
}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

if LTX_UPSAMPLER is not None:
    LTX_UPSAMPLER.to("cpu")
for component_name in ("transformer", "vae", "text_encoder"):
    component = (
        getattr(LTX_PIPE, component_name, None)
        if LTX_PIPE is not None
        else None
    )
    if component is not None and callable(getattr(component, "to", None)):
        try:
            component.to("cpu")
        except (RuntimeError, ValueError):
            pass
del LTX_PIPE, LTX_UPSCALE_PIPE, LTX_UPSAMPLER
gc.collect()
torch.cuda.empty_cache()

print({
    "final_video": str(LTX_FINAL_VIDEO_PATH),
    "manifest": str(LTX_FINAL_MANIFEST),
    "duration_seconds": round(
        len(LTX_JOBS) * (LTX_NUM_FRAMES - 1) / LTX_FPS,
        3,
    ),
    "cyclic_closure": (
        "last gap approaches the first canonical endpoint; "
        "no duplicated boundary frame"
    ),
})
display(Video(
    filename=str(LTX_FINAL_VIDEO_PATH),
    embed=False,
    width=LTX_DISPLAY_WIDTH,
))
if DOWNLOAD_LTX_FINAL_VIDEO:
    from google.colab import files
    files.download(str(LTX_FINAL_VIDEO_PATH))
